# ACDC Final SDF INR Inference and Figures

Inference-only notebook for the signed-distance INR model from `final.ipynb`.
It loads the final SDF checkpoint, extracts ED and ES ACDC contours, runs the
model, and creates figure-ready visualisations. There are no training,
cache-building, or test-set evaluation cells here.


## 1. Setup


In [ ]:
%pip install -q nibabel scikit-image scipy scikit-learn trimesh plotly rtree


In [ ]:
import os, re, math, time, warnings, configparser, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

import torch
import torch.nn as nn
import torch.nn.functional as F

import nibabel as nib
from skimage.measure import find_contours, marching_cubes
from scipy.ndimage import gaussian_filter
from sklearn.neighbors import KDTree
import trimesh

os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name} ({p.total_memory/1e9:.1f} GB)")


## 2. Configuration


In [ ]:
MODE = "combined"

CFG = dict(
    mode=MODE,
    input_dim=5 if MODE == "combined" else 4,
    latent_dim=256,
    fourier_L=3,
    decoder_hidden=512,
    decoder_layers=8,
    skip_layer=4,
    sphere_r0=0.5,
    decoder_activation="softplus",
    decoder_softplus_beta=100.0,
    decoder_spectral_norm=False,
    tau_min_norm=0.05,
    delta_cap_norm=0.45,
    grid_res=128,
    iso_level=0.0,
    bbox_pad=0.30,
    mc_smooth_sigma=0.0,
    mc_topology_cleanup=True,
    mc_cleanup_smooth_sigma=0.35,
    mc_slice_cleanup=False,
    mc_slice_min_pixels=12,
    mc_min_component_faces=64,
    mc_min_face_frac=0.02,
    mc_component_dist_weight=8.0,
    # Slice-lofted SDF prior: keeps every annotated SAX ring as a boundary
    # and interpolates the wall smoothly between slices. This replaces the
    # older contour-point sphere injection, which created circular artifacts.
    mc_slice_loft_weight=0.90,
    mc_slice_anchor_band_vox=1.25,
    mc_slice_anchor_weight=1.0,
    mc_slice_smooth_xy_sigma=0.25,
    mc_slice_smooth_z_sigma=0.0,
    mc_loft_ring_points=60,
    mc_min_3d_component_voxels=50,
)

CKPT_CANDIDATES = [
    Path("/kaggle/input/models/andrefce/sdf-final-final/pytorch/default/1/inr_sdf_combined_final(1).pt"),
    Path("/kaggle/working/inr_sdf_combined_continued_best.pt"),
    Path("/kaggle/working/inr_sdf_combined_continued_final.pt"),
    Path("inr_sdf_combined_continued_best.pt"),
    Path("inr_sdf_combined_continued_final.pt"),
]
CKPT_PATH = next((p for p in CKPT_CANDIDATES if p.exists()), CKPT_CANDIDATES[0])

ACDC_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/andrefce/realmri/training"),
    Path("/kaggle/input/realmri/training"),
    Path("/kaggle/input/acdc/training"),
]
ACDC_PATIENT_DIR = Path("/kaggle/input/datasets/andrefce/realmri/training/patient001")

def resolve_default_patient_dir(default_dir):
    if default_dir.exists():
        return default_dir
    for root in ACDC_ROOT_CANDIDATES:
        if root.exists():
            hits = sorted(p for p in root.glob("patient*") if p.is_dir())
            if hits:
                return hits[0]
    return default_dir

ACDC_PATIENT_DIR = resolve_default_patient_dir(ACDC_PATIENT_DIR)
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUT_DIR.mkdir(parents=True, exist_ok=True)

LBL_BG, LBL_RV, LBL_MYO, LBL_LV = 0, 1, 2, 3
N_PTS_PER_RING = 60
FLIP_Z = True
INFERENCE_GRID_RES = int(CFG["grid_res"])

print(f"Checkpoint: {CKPT_PATH}")
print(f"ACDC patient: {ACDC_PATIENT_DIR}")
print(f"Output dir: {OUT_DIR.resolve()}")
print(f"Grid: {INFERENCE_GRID_RES}^3 | input_dim={CFG['input_dim']} | fourier_L={CFG['fourier_L']}")


## 3. Final SDF Model


In [ ]:
class FourierPE(nn.Module):
    def __init__(self, L=6):
        super().__init__()
        self.L = L
        freqs = 2.0 ** torch.arange(L).float() * math.pi
        self.register_buffer("freqs", freqs)

    @property
    def out_dim(self):
        return 3 + 6 * self.L

    def forward(self, xyz):
        x = xyz.unsqueeze(-1) * self.freqs
        sin = torch.sin(x)
        cos = torch.cos(x)
        return torch.cat([xyz, sin.flatten(-2), cos.flatten(-2)], dim=-1)


class PointNetEncoder(nn.Module):
    def __init__(self, input_dim=4, latent_dim=256):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(inplace=True),
            nn.Linear(64, 128), nn.ReLU(inplace=True),
            nn.Linear(128, 256), nn.ReLU(inplace=True),
            nn.Linear(256, latent_dim),
        )
        self.proj = nn.Linear(latent_dim * 2, latent_dim)
        nn.init.normal_(self.proj.weight, 0.0, 0.01)
        nn.init.zeros_(self.proj.bias)

    def forward(self, x, mask):
        f = self.mlp(x)
        neg_inf = torch.finfo(f.dtype).min
        tissue = x[:, :, 3]
        endo_mask = mask & (tissue < 0.5)
        epi_mask = mask & (tissue >= 0.5)

        f_endo = f.masked_fill(~endo_mask.unsqueeze(-1), neg_inf)
        f_epi = f.masked_fill(~epi_mask.unsqueeze(-1), neg_inf)
        z_endo = f_endo.max(dim=1).values
        z_epi = f_epi.max(dim=1).values

        f_global = f.masked_fill(~mask.unsqueeze(-1), neg_inf)
        z_global = f_global.max(dim=1).values
        has_endo = endo_mask.any(dim=1, keepdim=True).float()
        has_epi = epi_mask.any(dim=1, keepdim=True).float()
        z_endo = z_endo * has_endo + z_global * (1.0 - has_endo)
        z_epi = z_epi * has_epi + z_global * (1.0 - has_epi)
        return self.proj(torch.cat([z_endo, z_epi], dim=-1))


class INRDecoderSDF(nn.Module):
    def __init__(self, latent_dim=256, fourier_L=6, hidden=512,
                 n_layers=8, skip_layer=4, r0=0.5,
                 delta_init_norm=0.28, delta_cap=None,
                 activation="relu", softplus_beta=100.0,
                 spectral_norm=False):
        super().__init__()
        self.skip_layer = skip_layer
        self.tau_min = float(delta_init_norm)
        self.delta_cap = None if delta_cap is None else float(delta_cap)
        if activation == "softplus":
            beta = float(softplus_beta)
            self._act = lambda x: F.softplus(x, beta=beta, threshold=20.0)
        else:
            self._act = lambda x: F.relu(x, inplace=False)

        in_dim = latent_dim + 3 + 6 * fourier_L
        self.in_proj = nn.Linear(in_dim, hidden)
        self.layers = nn.ModuleList()
        for li in range(n_layers):
            d_in = hidden + (in_dim if li == skip_layer else 0)
            self.layers.append(nn.Linear(d_in, hidden))
        self.head_endo = nn.Linear(hidden, 1)
        self.head_delta = nn.Linear(hidden, 1)

        with torch.no_grad():
            for m in self.layers:
                nn.init.normal_(m.weight, 0.0, math.sqrt(2.0 / m.in_features))
                nn.init.zeros_(m.bias)
            nn.init.normal_(self.head_endo.weight, 0.0, math.sqrt(math.pi) / math.sqrt(hidden))
            nn.init.constant_(self.head_endo.bias, -r0)
            nn.init.normal_(self.head_delta.weight, 0.0, 1e-2)
            if self.delta_cap is None:
                d = max(delta_init_norm - 1e-4, 1e-3)
                self.head_delta.bias.data.fill_(math.log(math.expm1(d)))
            else:
                self.head_delta.bias.data.fill_(0.0)

        if spectral_norm:
            self.in_proj = nn.utils.spectral_norm(self.in_proj)
            self.layers = nn.ModuleList([nn.utils.spectral_norm(m) for m in self.layers])

    def forward(self, z, fxyz):
        B, N, _ = fxyz.shape
        z_exp = z.unsqueeze(1).expand(B, N, -1)
        h_in = torch.cat([z_exp, fxyz], dim=-1)
        h = self._act(self.in_proj(h_in))
        for li, lyr in enumerate(self.layers):
            if li == self.skip_layer:
                h = torch.cat([h, h_in], dim=-1)
            h = self._act(lyr(h))
        f_endo = self.head_endo(h).squeeze(-1)
        raw_d = self.head_delta(h).squeeze(-1)
        if self.delta_cap is None:
            delta = F.softplus(raw_d) + 1e-4
        else:
            delta = self.tau_min + (self.delta_cap - self.tau_min) * torch.sigmoid(raw_d)
        return f_endo, f_endo - delta, delta


class SDFNetwork(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.encoder = PointNetEncoder(input_dim=cfg["input_dim"], latent_dim=cfg["latent_dim"])
        self.fourier = FourierPE(L=cfg["fourier_L"])
        self.decoder = INRDecoderSDF(
            latent_dim=cfg["latent_dim"],
            fourier_L=cfg["fourier_L"],
            hidden=cfg["decoder_hidden"],
            n_layers=cfg["decoder_layers"],
            skip_layer=cfg["skip_layer"],
            r0=cfg["sphere_r0"],
            delta_init_norm=cfg["tau_min_norm"],
            delta_cap=cfg.get("delta_cap_norm", None),
            activation=cfg.get("decoder_activation", "relu"),
            softplus_beta=cfg.get("decoder_softplus_beta", 100.0),
            spectral_norm=cfg.get("decoder_spectral_norm", False),
        )

    def encode(self, contour, mask):
        return self.encoder(contour, mask)

    def decode(self, z, query_xyz):
        return self.decoder(z, self.fourier(query_xyz))


def torch_load_checkpoint(path):
    try:
        return torch.load(path, map_location=DEVICE, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=DEVICE)


def strip_dataparallel_prefix(state):
    if not isinstance(state, dict):
        return state
    keys = list(state.keys())
    if keys and all(isinstance(k, str) and k.startswith("module.") for k in keys):
        return {k[len("module."):]: v for k, v in state.items()}
    return state


if not CKPT_PATH.exists():
    raise FileNotFoundError(
        f"Checkpoint not found: {CKPT_PATH}\n"
        "Edit CKPT_PATH in the configuration cell to point at the final.ipynb SDF checkpoint."
    )

ckpt = torch_load_checkpoint(CKPT_PATH)
ckpt_cfg = ckpt.get("cfg", {}) if isinstance(ckpt, dict) else {}
for key in [
    "input_dim", "latent_dim", "fourier_L", "decoder_hidden", "decoder_layers",
    "skip_layer", "sphere_r0", "tau_min_norm", "delta_cap_norm",
    "decoder_activation", "decoder_softplus_beta", "decoder_spectral_norm",
]:
    if key in ckpt_cfg:
        CFG[key] = ckpt_cfg[key]

model = SDFNetwork(CFG).to(DEVICE)
state = ckpt.get("model_state", ckpt) if isinstance(ckpt, dict) else ckpt
model.load_state_dict(strip_dataparallel_prefix(state), strict=True)
model.eval()

print(f"Loaded SDF checkpoint: {CKPT_PATH}")
print(f"Epoch: {ckpt.get('epoch', '?') if isinstance(ckpt, dict) else '?'}")
print(f"Val loss: {ckpt.get('val_loss', float('nan')) if isinstance(ckpt, dict) else float('nan'):.4f}")
print(f"Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Architecture: input_dim={CFG['input_dim']}, fourier_L={CFG['fourier_L']}, activation={CFG['decoder_activation']}")


## 4. Load ACDC ED and ES Contours


In [ ]:
def resolve_nii_path(p):
    p = Path(p)
    if p.is_file():
        return p
    if p.is_dir():
        for pat in ("*.nii.gz", "*.nii"):
            hits = sorted(p.glob(pat))
            if hits:
                return hits[0]
    raise FileNotFoundError(f"No NIfTI file found at {p}")


def find_ed_es_frames(patient_dir):
    patient_dir = Path(patient_dir)
    if not patient_dir.exists():
        raise FileNotFoundError(f"ACDC patient directory not found: {patient_dir}")
    gt = {}
    for child in sorted(patient_dir.iterdir()):
        m = re.search(r"_frame(\d+)_gt", child.name)
        if m and (child.is_dir() or child.name.endswith((".nii.gz", ".nii"))):
            gt[int(m.group(1))] = child
    if len(gt) < 2:
        raise ValueError(f"Need at least two GT frames in {patient_dir}, found {len(gt)}")

    ed_f = es_f = None
    cfg_path = patient_dir / "Info.cfg"
    if cfg_path.exists():
        parser = configparser.ConfigParser()
        with open(cfg_path, "r", encoding="utf-8") as f:
            parser.read_string("[DEFAULT]\n" + f.read())
        try:
            ed_f = int(parser["DEFAULT"]["ED"])
            es_f = int(parser["DEFAULT"]["ES"])
        except (KeyError, ValueError):
            ed_f = es_f = None

    if ed_f not in gt or es_f not in gt:
        vols = {}
        for frame_no, gp in gt.items():
            vol = nib.load(str(resolve_nii_path(gp))).get_fdata(dtype=np.float32)
            if vol.ndim == 4:
                vol = vol[..., 0]
            vols[frame_no] = int((vol == LBL_LV).sum())
        nonzero = {k: v for k, v in vols.items() if v > 0}
        ed_f = max(nonzero, key=nonzero.get)
        es_f = min(nonzero, key=nonzero.get)
    return gt[ed_f], gt[es_f], ed_f, es_f


def contour_area(pts):
    if len(pts) < 3:
        return 0.0
    x, y = pts[:, 0], pts[:, 1]
    return 0.5 * abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))


def sample_contour(pts2d, n=N_PTS_PER_RING):
    pts2d = np.asarray(pts2d, dtype=np.float32)
    if len(pts2d) < 3:
        return pts2d
    if np.linalg.norm(pts2d[0] - pts2d[-1]) > 1e-6:
        pts2d = np.vstack([pts2d, pts2d[0]])
    d = np.sqrt((np.diff(pts2d, axis=0) ** 2).sum(axis=1))
    total = float(d.sum())
    if total < 1e-6:
        return pts2d[:-1]
    t_old = np.concatenate([[0.0], np.cumsum(d)])
    t_new = np.linspace(0.0, total, int(n), endpoint=False)
    rows = np.interp(t_new, t_old, pts2d[:, 0])
    cols = np.interp(t_new, t_old, pts2d[:, 1])
    return np.column_stack([rows, cols]).astype(np.float32)


def load_and_extract_contours(gt_path):
    real = resolve_nii_path(gt_path)
    nii = nib.load(str(real))
    data = nii.get_fdata().astype(np.int32)
    if data.ndim == 4:
        data = data[..., 0]
    affine = nii.affine
    dz = float(nii.header.get_zooms()[2])
    support = sorted(
        s for s in range(data.shape[2])
        if (data[:, :, s] == LBL_LV).any() or (data[:, :, s] == LBL_MYO).any()
    )

    def vox2world(rows, cols, s):
        vox = np.column_stack([cols, rows, np.zeros(len(rows)), np.ones(len(rows))])
        world = (affine @ vox.T).T
        world[:, 2] = s * dz
        return world[:, :3]

    pts = []
    for s in support:
        seg = data[:, :, s]
        for label, tissue_id in [((seg == LBL_LV), 0.0), (((seg == LBL_MYO) | (seg == LBL_LV)), 1.0)]:
            mask = label.astype(np.uint8)
            if mask.sum() <= 10:
                continue
            contours = find_contours(mask, 0.5)
            if not contours:
                continue
            ring = sample_contour(max(contours, key=contour_area))
            xyz = vox2world(ring[:, 0], ring[:, 1], s)
            pts.append(np.column_stack([xyz, np.full(len(ring), tissue_id, dtype=np.float32)]))

    raw = np.vstack(pts).astype(np.float32)
    xyz, tissue = raw[:, :3], raw[:, 3].astype(np.float32)
    centroid = xyz.mean(axis=0)
    xyz_c = xyz - centroid
    scale = float(np.linalg.norm(xyz_c[:, :2], axis=1).mean())
    if not np.isfinite(scale) or scale < 1e-6:
        scale = float(np.std(xyz_c) + 1e-6)
    xyz_n = (xyz_c / scale).astype(np.float32)
    if FLIP_Z:
        xyz_n[:, 2] = -xyz_n[:, 2]
    return dict(gt_path=real, xyz=xyz_n, tissue=tissue, centroid=centroid.astype(np.float32),
                scale=scale, slices=support, data=data, affine=affine, dz=dz)


ed_gt_path, es_gt_path, ed_frame_no, es_frame_no = find_ed_es_frames(ACDC_PATIENT_DIR)
ed_case = load_and_extract_contours(ed_gt_path)
es_case = load_and_extract_contours(es_gt_path)
ed_case.update(phase="ED", frame=ed_frame_no)
es_case.update(phase="ES", frame=es_frame_no)

for case in (ed_case, es_case):
    print(f"{case['phase']}: frame={case['frame']:02d} | points={len(case['xyz'])} | "
          f"slices={len(case['slices'])} | scale={case['scale']:.2f} mm | path={case['gt_path'].name}")


## 5. SDF Inference Helpers


In [ ]:
def chamfer_mm(pred_verts, gt_verts, scale=1.0):
    """Bidirectional Chamfer distance in mm between two vertex sets."""
    if len(pred_verts) == 0 or len(gt_verts) == 0:
        return float("nan")
    pred = np.asarray(pred_verts, dtype=np.float32)
    gt = np.asarray(gt_verts, dtype=np.float32)
    tree_gt = KDTree(gt)
    tree_pred = KDTree(pred)
    d_p2g, _ = tree_gt.query(pred, k=1)
    d_g2p, _ = tree_pred.query(gt, k=1)
    return float(0.5 * (d_p2g.mean() + d_g2p.mean()) * scale)


def _build_contour_tensor(contour_xyz, tissue_labels, cfg, phase_val):
    cont = np.column_stack([contour_xyz, tissue_labels]).astype(np.float32)
    if phase_val is not None and cfg["input_dim"] == 5:
        cont = np.column_stack([
            cont, np.full((len(cont), 1), phase_val, dtype=np.float32)
        ])
    cont_t = torch.from_numpy(cont).unsqueeze(0).to(DEVICE)
    mask_t = torch.ones(1, len(cont), dtype=torch.bool, device=DEVICE)
    return cont_t, mask_t


def _slice_residual_mm(mdl, z, contour_xyz, tissue_labels, scale):
    """Mean absolute SDF value on input contour points, in mm."""
    pts = torch.from_numpy(contour_xyz.astype(np.float32)).unsqueeze(0).to(DEVICE)
    with torch.no_grad(), torch.amp.autocast("cuda", enabled=False):
        fe, fp, _ = mdl.decode(z, pts)
    fe = fe[0].float().cpu().numpy()
    fp = fp[0].float().cpu().numpy()
    endo_m = tissue_labels == 0
    epi_m = tissue_labels == 1
    res_endo = float(np.abs(fe[endo_m]).mean()) if endo_m.any() else 0.0
    res_epi = float(np.abs(fp[epi_m]).mean()) if epi_m.any() else 0.0
    n = int(endo_m.sum() + epi_m.sum())
    res = (res_endo * endo_m.sum() + res_epi * epi_m.sum()) / max(1, n)
    return float(res * scale)


def _build_grid_and_query(z, mdl, contour_xyz, cfg, grid_res, batch_query):
    """Query the model SDF on a dense grid. No filtering, no field edits."""
    lo = contour_xyz.min(0) - cfg["bbox_pad"]
    hi = contour_xyz.max(0) + cfg["bbox_pad"]
    xs = np.linspace(lo[0], hi[0], grid_res)
    ys = np.linspace(lo[1], hi[1], grid_res)
    zs = np.linspace(lo[2], hi[2], grid_res)
    gx, gy, gz = np.meshgrid(xs, ys, zs, indexing="ij")
    grid_pts = np.stack([gx.ravel(), gy.ravel(), gz.ravel()], axis=-1).astype(np.float32)

    sdf_e = np.empty(len(grid_pts), np.float32)
    sdf_p = np.empty(len(grid_pts), np.float32)
    dlt = np.empty(len(grid_pts), np.float32)
    with torch.no_grad():
        for s in range(0, len(grid_pts), batch_query):
            chunk = torch.from_numpy(grid_pts[s:s + batch_query]).unsqueeze(0).to(DEVICE)
            with torch.amp.autocast("cuda", enabled=False):
                fe, fp, dl = mdl.decode(z, chunk)
            sdf_e[s:s + batch_query] = fe[0].float().cpu().numpy()
            sdf_p[s:s + batch_query] = fp[0].float().cpu().numpy()
            dlt[s:s + batch_query] = dl[0].float().cpu().numpy()

    shape = (grid_res, grid_res, grid_res)
    voxel = (hi - lo) / (grid_res - 1)
    return sdf_e.reshape(shape), sdf_p.reshape(shape), dlt.reshape(shape), lo, hi, voxel


def _mc_raw_no_cleanup(field, lo, voxel, iso_level):
    """Direct marching cubes on the raw model field.

    No Gaussian filtering, no boundary ramp, no padding shell, no component
    selection, no trimesh repair/process, no smoothing. If the raw field
    produces open or fragmented meshes, that is what we want to see.
    """
    field = np.asarray(field, dtype=np.float32)
    if float(field.min()) > iso_level or float(field.max()) < iso_level:
        return trimesh.Trimesh(process=False), True
    try:
        verts, faces, _, _ = marching_cubes(field, level=iso_level, spacing=tuple(voxel))
        verts = verts + lo
        mesh = trimesh.Trimesh(
            vertices=verts.astype(np.float32),
            faces=faces.astype(np.int64),
            process=False,
        )
        return mesh, False
    except Exception:
        return trimesh.Trimesh(process=False), True


# Alias for backwards compatibility (overridden by topology-cleanup cell)
_mc_raw = _mc_raw_no_cleanup


@torch.no_grad()
def predict_mesh_sdf(net, contour_xyz, tissue_labels, cfg,
                     grid_res=None, batch_query=131072, phase_val=None,
                     return_latent=False):
    """RAW model inference: model SDF grid -> direct marching cubes."""
    if grid_res is None:
        grid_res = cfg["grid_res"]
    mdl = net.module if isinstance(net, nn.DataParallel) else net
    mdl.eval()

    cont_t, mask_t = _build_contour_tensor(contour_xyz, tissue_labels, cfg, phase_val)
    z = mdl.encode(cont_t, mask_t)
    sdf_e, sdf_p, dlt, lo, hi, voxel = _build_grid_and_query(
        z, mdl, contour_xyz, cfg, grid_res, batch_query
    )
    endo, deg_e = _mc_raw(sdf_e, lo, voxel, cfg["iso_level"])
    epi, deg_p = _mc_raw(sdf_p, lo, voxel, cfg["iso_level"])

    if return_latent:
        return endo, epi, sdf_e, sdf_p, dlt, z, lo, hi, voxel, bool(deg_e or deg_p)
    return endo, epi, sdf_e, sdf_p, dlt


def predict_mesh_sdf_tto(net, contour_xyz, tissue_labels, cfg,
                         grid_res=None, batch_query=131072, phase_val=None,
                         scale=1.0, tto_steps=0, tto_lr=0.0,
                         tto_lambda_z=0.0, tto_lambda_eik=0.0,
                         tto_lambda_sign=0.0, use_tto=False):
    """Compatibility wrapper. TTO is intentionally disabled.

    Visualisation cells still call this name, but it now returns raw model
    output only.
    """
    mdl = net.module if isinstance(net, nn.DataParallel) else net
    out = predict_mesh_sdf(
        net, contour_xyz, tissue_labels, cfg,
        grid_res=grid_res, batch_query=batch_query,
        phase_val=phase_val, return_latent=True,
    )
    endo, epi, sdf_e, sdf_p, dlt, z, lo, hi, voxel, degenerate = out
    res = _slice_residual_mm(mdl, z, contour_xyz, tissue_labels, scale)
    return dict(
        endo=endo, epi=epi,
        sdf_e=sdf_e, sdf_p=sdf_p, dlt=dlt,
        lo=lo, hi=hi, voxel=voxel,
        z=z, z0=z,
        slice_res_mm_before=res,
        slice_res_mm_after=res,
        tto_history=[],
        degenerate=degenerate,
    )


@torch.no_grad()
def wt_at_endo_vertices(net, contour_xyz, tissue_labels, endo_verts, cfg,
                        phase_val=None, z_override=None):
    """Analytic wall thickness at raw endo-mesh vertices."""
    if len(endo_verts) == 0:
        return np.zeros(0, np.float32)
    mdl = net.module if isinstance(net, nn.DataParallel) else net
    mdl.eval()
    if z_override is None:
        cont_t, mask_t = _build_contour_tensor(contour_xyz, tissue_labels, cfg, phase_val)
        z = mdl.encode(cont_t, mask_t)
    else:
        z = z_override
    pts = torch.from_numpy(endo_verts.astype(np.float32)).unsqueeze(0).to(DEVICE)
    with torch.amp.autocast("cuda", enabled=False):
        _, _, dl = mdl.decode(z, pts)
    return dl[0].float().cpu().numpy()


print("RAW inference helpers defined: no filtering, repair, component selection, smoothing, or TTO")


In [ ]:
# Slice-lofted marching-cubes override.
#
# The previous topology cleanup forced balls around contour points to be inside
# the mask. Contour points are zero-level boundaries, not interior samples, so
# that created circular / beaded artifacts between sparse SAX slices, especially
# for the small ES cavity.
#
# This version builds a contour-loft signed-distance prior instead:
#   - each annotated ring stays a 2D zero boundary on its SAX slice,
#   - rings are interpolated along z to make smooth walls between slices,
#   - whole annotated slice bands are anchored to the loft prior,
#   - the raw network SDF is blended in away from those anchors.
from scipy.ndimage import binary_fill_holes, distance_transform_edt
from scipy.ndimage import generate_binary_structure, label as ndi_label
from matplotlib.path import Path as MplPath

_MC_STRUCT = generate_binary_structure(3, 2)  # 26-connectivity

CFG.setdefault('mc_topology_cleanup', True)
CFG.setdefault('mc_cleanup_smooth_sigma', 0.35)
CFG.setdefault('mc_slice_cleanup', False)
CFG.setdefault('mc_slice_min_pixels', 12)
CFG.setdefault('mc_min_component_faces', 64)
CFG.setdefault('mc_component_dist_weight', 8.0)
CFG.setdefault('mc_slice_loft_weight', 0.90)
CFG.setdefault('mc_slice_anchor_band_vox', 1.25)
CFG.setdefault('mc_slice_anchor_weight', 1.0)
CFG.setdefault('mc_slice_smooth_xy_sigma', 0.25)
CFG.setdefault('mc_slice_smooth_z_sigma', 0.0)
CFG.setdefault('mc_loft_ring_points', int(globals().get('N_PTS_PER_RING', 60)))
CFG.setdefault('mc_min_3d_component_voxels', 50)


def _empty_mesh():
    return trimesh.Trimesh(
        vertices=np.empty((0, 3), np.float32),
        faces=np.empty((0, 3), np.int64),
        process=False,
    )


def _contour_points_for_tissue(contour_xyz, tissue_labels, tissue_id):
    if contour_xyz is None or tissue_labels is None or tissue_id is None:
        return np.empty((0, 3), np.float32)
    m = np.asarray(tissue_labels) == tissue_id
    if not np.any(m):
        return np.empty((0, 3), np.float32)
    return np.asarray(contour_xyz, dtype=np.float32)[m]


def _resample_ring_xy(ring, n):
    ring = np.asarray(ring, dtype=np.float64)
    if len(ring) < 3:
        return ring.astype(np.float32)
    if np.linalg.norm(ring[0] - ring[-1]) > 1e-8:
        ring = np.vstack([ring, ring[0]])
    seg = np.sqrt((np.diff(ring, axis=0) ** 2).sum(axis=1))
    total = float(seg.sum())
    if total < 1e-8:
        return ring[:-1].astype(np.float32)
    t_old = np.concatenate([[0.0], np.cumsum(seg)])
    t_new = np.linspace(0.0, total, int(n), endpoint=False)
    x = np.interp(t_new, t_old, ring[:, 0])
    y = np.interp(t_new, t_old, ring[:, 1])
    return np.column_stack([x, y]).astype(np.float32)


def _angle_sort_ring_xy(xy):
    xy = np.asarray(xy, dtype=np.float64)
    c = xy.mean(axis=0)
    ang = np.arctan2(xy[:, 1] - c[1], xy[:, 0] - c[0])
    return xy[np.argsort(ang)]


def _align_ring_to_reference(ring, ref):
    ring = np.asarray(ring, dtype=np.float32)
    ref = np.asarray(ref, dtype=np.float32)
    if len(ring) != len(ref) or len(ring) == 0:
        return ring

    candidates = (ring, ring[::-1].copy())
    best_ring, best_score = ring, np.inf
    for cand in candidates:
        for shift in range(len(cand)):
            rolled = np.roll(cand, shift, axis=0)
            score = float(np.mean((rolled - ref) ** 2))
            if score < best_score:
                best_ring = rolled
                best_score = score
    return best_ring.astype(np.float32)


def _rings_by_slice_for_tissue(contour_xyz, tissue_labels, tissue_id, cfg):
    pts = _contour_points_for_tissue(contour_xyz, tissue_labels, tissue_id)
    if len(pts) < 3:
        return []

    n_ring = int(cfg.get('mc_loft_ring_points', globals().get('N_PTS_PER_RING', 60)))
    z_round = np.round(pts[:, 2].astype(np.float64), 6)
    rings = []
    for z_key in sorted(np.unique(z_round)):
        group = pts[z_round == z_key]
        if len(group) < 3:
            continue
        xy = _angle_sort_ring_xy(group[:, :2])
        ring = _resample_ring_xy(xy, n_ring)
        if len(ring) < 3:
            continue
        rings.append((float(group[:, 2].mean()), ring))

    rings.sort(key=lambda item: item[0])
    if not rings:
        return []

    aligned = [rings[0]]
    ref = rings[0][1]
    for z, ring in rings[1:]:
        ring = _align_ring_to_reference(ring, ref)
        aligned.append((z, ring))
        ref = ring
    return aligned


def _interpolate_ring_at_z(rings, z):
    if not rings:
        return None
    if len(rings) == 1:
        return rings[0][1]

    zs = np.asarray([r[0] for r in rings], dtype=np.float64)
    z = float(z)
    if z < zs[0] or z > zs[-1]:
        return None
    hi = int(np.searchsorted(zs, z, side='right'))
    if hi == 0:
        return rings[0][1]
    if hi >= len(rings):
        return rings[-1][1]
    z0, z1 = zs[hi - 1], zs[hi]
    t = 0.0 if abs(z1 - z0) < 1e-8 else (z - z0) / (z1 - z0)
    return ((1.0 - t) * rings[hi - 1][1] + t * rings[hi][1]).astype(np.float32)


def _signed_distance_from_ring(ring_xy, xs, ys, voxel_xy):
    nx, ny = len(xs), len(ys)
    gx, gy = np.meshgrid(xs, ys, indexing='ij')
    q = np.column_stack([gx.ravel(), gy.ravel()])
    inside = MplPath(np.asarray(ring_xy, dtype=np.float64), closed=True)
    inside = inside.contains_points(q).reshape(nx, ny)

    # Negative inside, positive outside: matches the SDF convention used by MC.
    outside = distance_transform_edt(~inside, sampling=tuple(voxel_xy))
    inside_d = distance_transform_edt(inside, sampling=tuple(voxel_xy))
    return (outside - inside_d).astype(np.float32)


def _slice_anchor_weights(shape, lo, voxel, rings, cfg):
    if not rings:
        return None
    band_vox = float(cfg.get('mc_slice_anchor_band_vox', 1.25) or 0.0)
    strength = float(cfg.get('mc_slice_anchor_weight', 1.0) or 0.0)
    if band_vox <= 0 or strength <= 0:
        return None
    zs = lo[2] + np.arange(shape[2], dtype=np.float64) * float(voxel[2])
    slice_z = np.asarray([r[0] for r in rings], dtype=np.float64)
    dz = np.abs(zs[:, None] - slice_z[None, :]).min(axis=1)
    band = max(band_vox * float(voxel[2]), 1e-8)
    wz = np.clip(1.0 - dz / band, 0.0, 1.0) ** 2
    wz = (strength * wz).astype(np.float32)
    return wz.reshape(1, 1, -1)


def _loft_sdf_from_contours(contour_xyz, tissue_labels, tissue_id, lo, voxel, shape, cfg):
    rings = _rings_by_slice_for_tissue(contour_xyz, tissue_labels, tissue_id, cfg)
    if not rings:
        return None, []

    xs = lo[0] + np.arange(shape[0], dtype=np.float64) * float(voxel[0])
    ys = lo[1] + np.arange(shape[1], dtype=np.float64) * float(voxel[1])
    zs = lo[2] + np.arange(shape[2], dtype=np.float64) * float(voxel[2])

    bbox_diag = float(np.linalg.norm(np.asarray(shape, dtype=np.float64) * np.asarray(voxel, dtype=np.float64)))
    outside_value = max(bbox_diag, 1.0)
    loft = np.full(shape, outside_value, dtype=np.float32)

    z0, z1 = rings[0][0], rings[-1][0]
    for k, z in enumerate(zs):
        if z < z0 or z > z1:
            continue
        ring = _interpolate_ring_at_z(rings, z)
        if ring is not None:
            loft[:, :, k] = _signed_distance_from_ring(ring, xs, ys, voxel[:2])

    return loft, rings


def _select_inside_component(inside, lo, voxel, contour_pts):
    """Keep the connected component that best overlaps the contour evidence."""
    labels, n_comp = ndi_label(inside, structure=_MC_STRUCT)
    if n_comp <= 1:
        return inside.astype(bool, copy=False)

    sizes = np.bincount(labels.ravel()).astype(np.float64)
    sizes[0] = 0.0
    votes = np.zeros_like(sizes)

    if contour_pts is not None and len(contour_pts):
        dims = np.asarray(labels.shape, dtype=np.int64)
        ijk = np.rint((np.asarray(contour_pts) - lo) / voxel).astype(np.int64)
        ijk = np.clip(ijk, 0, dims - 1)
        vote_radius = 2
        for ix, iy, iz in ijk:
            nb = labels[
                max(0, ix - vote_radius):min(dims[0], ix + vote_radius + 1),
                max(0, iy - vote_radius):min(dims[1], iy + vote_radius + 1),
                max(0, iz - vote_radius):min(dims[2], iz + vote_radius + 1),
            ].ravel()
            nb = nb[nb > 0]
            if len(nb):
                lab, cnt = np.unique(nb, return_counts=True)
                votes[lab] += cnt

    if votes.max() > 0:
        score = sizes / max(sizes.max(), 1.0) + 4.0 * votes / votes.max()
    else:
        score = sizes
    keep = int(np.argmax(score[1:]) + 1)
    return labels == keep


def _clean_inside_3d(inside, cfg):
    """Fallback cleanup only: remove tiny 3D islands and fill internal holes.

    No binary opening is used here because it can erase the thin ES cavity/wall
    and create the very slice-to-slice discontinuities we are trying to avoid.
    """
    min_vox = int(cfg.get('mc_min_3d_component_voxels', 50))
    labels, n_comp = ndi_label(inside, structure=_MC_STRUCT)
    if n_comp > 1:
        sizes = np.bincount(labels.ravel())
        sizes[0] = 0
        inside = (sizes >= min_vox)[labels]
    return binary_fill_holes(inside, structure=_MC_STRUCT).astype(bool)


def _fallback_clean_field(field, lo, voxel, iso_level, contour_xyz, tissue_labels, tissue_id, cfg):
    inside = np.asarray(field <= iso_level, dtype=bool)
    inside[[0, -1], :, :] = False
    inside[:, [0, -1], :] = False
    inside[:, :, [0, -1]] = False
    if (not inside.any()) or inside.all():
        return field.astype(np.float32, copy=False)

    pts = _contour_points_for_tissue(contour_xyz, tissue_labels, tissue_id)
    inside = _select_inside_component(inside, lo, voxel, pts)
    inside = _clean_inside_3d(inside, cfg)
    inside[[0, -1], :, :] = False
    inside[:, [0, -1], :] = False
    inside[:, :, [0, -1]] = False
    if (not inside.any()) or inside.all():
        return field.astype(np.float32, copy=False)

    outside = distance_transform_edt(~inside, sampling=tuple(voxel))
    inside_d = distance_transform_edt(inside, sampling=tuple(voxel))
    clean = (outside - inside_d).astype(np.float32)

    sigma = float(cfg.get('mc_cleanup_smooth_sigma', 0.0) or 0.0)
    if sigma > 0:
        clean = gaussian_filter(clean, sigma=sigma).astype(np.float32)
    return clean


def _topology_clean_field(field, lo, voxel, iso_level, contour_xyz,
                          tissue_labels, tissue_id, cfg):
    """Build a smooth, slice-anchored field before marching cubes."""
    field = np.nan_to_num(np.asarray(field, dtype=np.float32),
                          nan=1e3, posinf=1e3, neginf=-1e3)
    if not cfg.get('mc_topology_cleanup', True):
        return field.astype(np.float32, copy=False)

    loft, rings = _loft_sdf_from_contours(
        contour_xyz, tissue_labels, tissue_id, lo, voxel, field.shape, cfg
    )
    if loft is None:
        return _fallback_clean_field(field, lo, voxel, iso_level,
                                     contour_xyz, tissue_labels, tissue_id, cfg)

    alpha = float(np.clip(cfg.get('mc_slice_loft_weight', 0.90), 0.0, 1.0))
    clean = ((1.0 - alpha) * field + alpha * loft).astype(np.float32)

    anchor_w = _slice_anchor_weights(field.shape, lo, voxel, rings, cfg)
    if anchor_w is not None:
        clean = ((1.0 - anchor_w) * clean + anchor_w * loft).astype(np.float32)

    sigma_xy = float(cfg.get('mc_slice_smooth_xy_sigma', 0.25) or 0.0)
    sigma_z = float(cfg.get('mc_slice_smooth_z_sigma', 0.0) or 0.0)
    if sigma_xy > 0 or sigma_z > 0:
        clean = gaussian_filter(clean, sigma=(sigma_xy, sigma_xy, sigma_z)).astype(np.float32)
        # Re-anchor after smoothing so the visible slice intersections remain
        # tied to the actual SAX contours.
        if anchor_w is not None:
            clean = ((1.0 - anchor_w) * clean + anchor_w * loft).astype(np.float32)

    return clean.astype(np.float32)


def _single_mesh_component(mesh, contour_xyz, tissue_labels, tissue_id, cfg):
    if mesh is None or len(mesh.faces) == 0 or len(mesh.vertices) == 0:
        return _empty_mesh()
    try:
        parts = list(mesh.split(only_watertight=False))
    except Exception:
        parts = [mesh]
    parts = [p for p in parts if len(p.faces) > 0 and len(p.vertices) > 0]
    if len(parts) <= 1:
        out = parts[0].copy() if parts else _empty_mesh()
        if len(out.vertices):
            out.remove_unreferenced_vertices()
        return out

    pts = _contour_points_for_tissue(contour_xyz, tissue_labels, tissue_id)
    diag = float(np.linalg.norm(np.asarray(contour_xyz).max(0) - np.asarray(contour_xyz).min(0))) if contour_xyz is not None and len(contour_xyz) else 1.0
    diag = max(diag, 1e-6)
    max_faces = max(len(p.faces) for p in parts)
    min_faces = max(
        int(cfg.get('mc_min_component_faces', 64)),
        int(max_faces * float(cfg.get('mc_min_face_frac', 0.0) or 0.0)),
    )
    dist_w = float(cfg.get('mc_component_dist_weight', 8.0) or 0.0)

    best_i, best_score = 0, -np.inf
    for i, part in enumerate(parts):
        score = math.log1p(len(part.faces))
        if len(part.faces) < min_faces:
            score -= 2.0
        if len(pts):
            try:
                d, _ = KDTree(np.asarray(part.vertices, dtype=np.float32)).query(pts, k=1)
                score -= dist_w * float(np.median(d)) / diag
            except Exception:
                pass
        if score > best_score:
            best_i, best_score = i, score

    out = parts[best_i].copy()
    out.remove_unreferenced_vertices()
    try:
        out.fix_normals()
    except Exception:
        pass
    return out


def _mc_single_surface(field, lo, voxel, iso_level, contour_xyz=None,
                       tissue_labels=None, tissue_id=None, cfg=None):
    cfg = CFG if cfg is None else cfg
    field = np.nan_to_num(np.asarray(field, dtype=np.float32),
                          nan=1e3, posinf=1e3, neginf=-1e3)

    sigma = float(cfg.get('mc_smooth_sigma', 0.0) or 0.0)
    work = gaussian_filter(field, sigma=sigma).astype(np.float32) if sigma > 0 else field
    work = _topology_clean_field(work, lo, voxel, iso_level,
                                 contour_xyz, tissue_labels, tissue_id, cfg)

    if float(work.min()) > iso_level or float(work.max()) < iso_level:
        return _empty_mesh(), True, work
    try:
        verts, faces, _, _ = marching_cubes(work, level=iso_level, spacing=tuple(voxel))
        mesh = trimesh.Trimesh(
            vertices=(verts + lo).astype(np.float32),
            faces=faces.astype(np.int64),
            process=False,
        )
        mesh = _single_mesh_component(mesh, contour_xyz, tissue_labels, tissue_id, cfg)
        return mesh, len(mesh.faces) == 0, work
    except Exception:
        return _empty_mesh(), True, work


def _mc_raw(field, lo, voxel, iso_level):
    mesh, deg, _ = _mc_single_surface(field, lo, voxel, iso_level, cfg=CFG)
    return mesh, deg


@torch.no_grad()
def predict_mesh_sdf(net, contour_xyz, tissue_labels, cfg,
                     grid_res=None, batch_query=131072, phase_val=None,
                     return_latent=False):
    if grid_res is None:
        grid_res = cfg['grid_res']
    mdl = net.module if isinstance(net, nn.DataParallel) else net
    mdl.eval()

    cont_t, mask_t = _build_contour_tensor(contour_xyz, tissue_labels, cfg, phase_val)
    z = mdl.encode(cont_t, mask_t)
    raw_e, raw_p, dlt, lo, hi, voxel = _build_grid_and_query(
        z, mdl, contour_xyz, cfg, grid_res, batch_query
    )
    endo, deg_e, sdf_e = _mc_single_surface(
        raw_e, lo, voxel, cfg['iso_level'], contour_xyz, tissue_labels, 0, cfg
    )
    epi, deg_p, sdf_p = _mc_single_surface(
        raw_p, lo, voxel, cfg['iso_level'], contour_xyz, tissue_labels, 1, cfg
    )

    if return_latent:
        return endo, epi, sdf_e, sdf_p, dlt, z, lo, hi, voxel, bool(deg_e or deg_p), raw_e, raw_p
    return endo, epi, sdf_e, sdf_p, dlt


def predict_mesh_sdf_tto(net, contour_xyz, tissue_labels, cfg,
                         grid_res=None, batch_query=131072, phase_val=None,
                         scale=1.0, tto_steps=0, tto_lr=0.0,
                         tto_lambda_z=0.0, tto_lambda_eik=0.0,
                         tto_lambda_sign=0.0, use_tto=False):
    mdl = net.module if isinstance(net, nn.DataParallel) else net
    out = predict_mesh_sdf(
        net, contour_xyz, tissue_labels, cfg,
        grid_res=grid_res, batch_query=batch_query,
        phase_val=phase_val, return_latent=True,
    )
    if len(out) == 12:
        endo, epi, sdf_e, sdf_p, dlt, z, lo, hi, voxel, degenerate, raw_e, raw_p = out
    else:
        endo, epi, sdf_e, sdf_p, dlt, z, lo, hi, voxel, degenerate = out
        raw_e, raw_p = sdf_e, sdf_p
    res = _slice_residual_mm(mdl, z, contour_xyz, tissue_labels, scale)
    return dict(
        endo=endo, epi=epi,
        sdf_e=sdf_e, sdf_p=sdf_p, raw_sdf_e=raw_e, raw_sdf_p=raw_p, dlt=dlt,
        lo=lo, hi=hi, voxel=voxel,
        z=z, z0=z,
        slice_res_mm_before=res,
        slice_res_mm_after=res,
        tto_history=[],
        degenerate=degenerate,
    )


print('Slice-lofted SDF inference helpers active (anchored slice rings, no contour-point spheres)')


## 6. Run Inference


In [ ]:
def infer_case(case, grid_res=INFERENCE_GRID_RES):
    phase_val = None
    if CFG["input_dim"] == 5:
        phase_val = 1.0 if case["phase"] == "ES" else 0.0
    t0 = time.time()
    out = predict_mesh_sdf(
        model, case["xyz"], case["tissue"], CFG,
        grid_res=grid_res, phase_val=phase_val, return_latent=True,
    )
    if len(out) == 12:
        endo, epi, sdf_e, sdf_p, dlt, z, lo, hi, voxel, degenerate, raw_e, raw_p = out
    else:
        endo, epi, sdf_e, sdf_p, dlt, z, lo, hi, voxel, degenerate = out
        raw_e, raw_p = sdf_e, sdf_p
    wt_norm = wt_at_endo_vertices(
        model, case["xyz"], case["tissue"], endo.vertices,
        CFG, phase_val=phase_val, z_override=z,
    )
    case.update(
        phase_val=phase_val, endo=endo, epi=epi,
        sdf_endo=sdf_e, sdf_epi=sdf_p, sdf_endo_raw=raw_e, sdf_epi_raw=raw_p, delta_grid=dlt,
        latent=z, grid_lo=lo, grid_hi=hi, voxel=voxel,
        degenerate=degenerate, wt_mm=wt_norm * case["scale"],
        elapsed_sec=time.time() - t0,
    )
    return case


print(f"Running final SDF model at grid={INFERENCE_GRID_RES}^3")
ed_case = infer_case(ed_case)
es_case = infer_case(es_case)

def mesh_volume_ml(mesh, scale_mm):
    if mesh is None or len(mesh.faces) == 0:
        return float("nan")
    try:
        return float(abs(mesh.volume) * (scale_mm ** 3) / 1000.0)
    except Exception:
        return float("nan")

summary_df = pd.DataFrame([
    dict(
        phase=case["phase"],
        frame=case["frame"],
        points=len(case["xyz"]),
        slices=len(case["slices"]),
        scale_mm=case["scale"],
        endo_faces=len(case["endo"].faces),
        epi_faces=len(case["epi"].faces),
        endo_volume_ml=mesh_volume_ml(case["endo"], case["scale"]),
        epi_volume_ml=mesh_volume_ml(case["epi"], case["scale"]),
        median_wt_mm=float(np.nanmedian(case["wt_mm"])) if len(case["wt_mm"]) else float("nan"),
        p05_wt_mm=float(np.nanpercentile(case["wt_mm"], 5)) if len(case["wt_mm"]) else float("nan"),
        p95_wt_mm=float(np.nanpercentile(case["wt_mm"], 95)) if len(case["wt_mm"]) else float("nan"),
        seconds=case["elapsed_sec"],
        degenerate=case["degenerate"],
    )
    for case in (ed_case, es_case)
])
display(summary_df.round(3))


## 7. Publication-Style Figure Helpers


In [ ]:
STYLE = {
    "font.family": "serif",
    "font.serif": ["DejaVu Serif", "Times New Roman", "Times"],
    "font.size": 10,
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
}

PALETTE = dict(
    endo="#1f77b4", epi="#d62728",
    ed="#2c7fb8", es="#d95f0e",
    contour_endo="#4db8ff", contour_epi="#ff6b6b",
)
SEG_CMAP = ListedColormap([
    (0.00, 0.00, 0.00, 0.00),
    (0.30, 0.50, 1.00, 1.00),
    (0.20, 0.80, 0.30, 1.00),
    (1.00, 0.30, 0.30, 1.00),
])

def save_figure(fig, name):
    png = OUT_DIR / f"{name}.png"
    pdf = OUT_DIR / f"{name}.pdf"
    fig.savefig(png, facecolor=fig.get_facecolor())
    fig.savefig(pdf, facecolor=fig.get_facecolor())
    print(f"Saved {png}")
    print(f"Saved {pdf}")

def plot_faces(mesh, max_faces=12000):
    if mesh is None or len(mesh.faces) == 0 or len(mesh.vertices) == 0:
        return None, None
    vertices = np.asarray(mesh.vertices, dtype=np.float32)
    faces = np.asarray(mesh.faces, dtype=np.int64)
    if len(faces) > max_faces:
        keep = np.linspace(0, len(faces) - 1, int(max_faces)).astype(np.int64)
        faces = faces[keep]
    return vertices, faces

def add_solid_mesh(ax, mesh, color, alpha=0.85, max_faces=12000):
    vertices, faces = plot_faces(mesh, max_faces=max_faces)
    if vertices is None:
        return
    ax.add_collection3d(Poly3DCollection(vertices[faces], facecolor=color, edgecolor="none", alpha=alpha))

def add_colored_mesh(ax, mesh, values, cmap="viridis", vmin=None, vmax=None, max_faces=14000):
    vertices, faces = plot_faces(mesh, max_faces=max_faces)
    if vertices is None or values is None or len(values) == 0:
        return None
    values = np.asarray(values, dtype=np.float32)
    face_values = values[faces].mean(axis=1)
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    cmap_fn = plt.get_cmap(cmap)
    ax.add_collection3d(Poly3DCollection(vertices[faces], facecolors=cmap_fn(norm(face_values)), edgecolor="none", alpha=0.95))
    return plt.cm.ScalarMappable(norm=norm, cmap=cmap_fn)

def set_3d_lim(ax, arrays, pad=0.08):
    pts = [np.asarray(a) for a in arrays if a is not None and len(a)]
    if not pts:
        return
    pts = np.vstack(pts)
    pts = pts[np.isfinite(pts).all(axis=1)]
    lo, hi = pts.min(axis=0), pts.max(axis=0)
    ctr = (lo + hi) * 0.5
    rad = max(float((hi - lo).max()) * (0.5 + pad), 1e-3)
    ax.set_xlim(ctr[0] - rad, ctr[0] + rad)
    ax.set_ylim(ctr[1] - rad, ctr[1] + rad)
    ax.set_zlim(ctr[2] - rad, ctr[2] + rad)
    ax.set_box_aspect([1, 1, 1])

def support_mid_slice(data):
    support = np.where(((data == LBL_LV) | (data == LBL_MYO)).any(axis=(0, 1)))[0]
    return int(support[len(support) // 2]) if len(support) else int(data.shape[2] // 2)

def mesh_to_world(mesh, centroid, scale, flip_z=FLIP_Z):
    """Transform mesh from normalised coords back to world mm.

    When flip_z is True, negating the Z axis reverses face winding.
    We compensate by flipping face vertex order so that normals remain
    outward-pointing and Plotly/trimesh render correctly.
    """
    if mesh is None or len(mesh.faces) == 0:
        return _empty_mesh()
    vertices = np.asarray(mesh.vertices, dtype=np.float64).copy()
    faces = np.asarray(mesh.faces, dtype=np.int64).copy()
    if flip_z:
        vertices[:, 2] = -vertices[:, 2]
        # Flipping one axis reverses winding → flip face order to restore normals
        faces = faces[:, ::-1]
    vertices = vertices * float(scale) + np.asarray(centroid, dtype=np.float64)
    return trimesh.Trimesh(vertices=vertices, faces=faces, process=False)

def mesh_section_xy(mesh, z_val):
    """Cut mesh with a horizontal plane at z=z_val and return 2D line segments."""
    if mesh is None or len(mesh.faces) == 0 or len(mesh.vertices) == 0:
        return []
    try:
        section = mesh.section(plane_origin=[0.0, 0.0, float(z_val)], plane_normal=[0.0, 0.0, 1.0])
    except Exception:
        return []
    if section is None:
        return []
    try:
        return [np.asarray(line)[:, :2] for line in section.discrete if len(line) >= 2]
    except Exception:
        return []

def selected_slice_z(case, n=6):
    """Select n representative Z levels from the case contour points."""
    z_values = np.sort(np.unique(np.round(case["xyz"][:, 2], 5)))
    if len(z_values) <= n:
        return z_values
    idx = np.linspace(0, len(z_values) - 1, int(n)).round().astype(int)
    return z_values[idx]

## 8. Figure 1 — Raw vs Slice-Lofted SDF Reconstruction

Side-by-side comparison of the direct marching-cubes output from the model
(raw SDF zero-crossing) against the slice-lofted, contour-anchored version.
This highlights the effect of using the annotated SAX rings as zero-boundary
constraints while smoothing the wall between slices.

In [ ]:
@torch.no_grad()
def infer_raw_meshes(case, grid_res=INFERENCE_GRID_RES):
    """Run raw marching cubes (no topology cleanup) on the already-computed SDF grids."""
    # We already have the SDF grids stored in case from infer_case()
    sdf_e = case.get("sdf_endo_raw", case["sdf_endo"])
    sdf_p = case.get("sdf_epi_raw", case["sdf_epi"])
    lo = case["grid_lo"]
    voxel = case["voxel"]
    iso = CFG["iso_level"]
    endo_raw, _ = _mc_raw_no_cleanup(sdf_e, lo, voxel, iso)
    epi_raw, _ = _mc_raw_no_cleanup(sdf_p, lo, voxel, iso)
    return endo_raw, epi_raw


def figure_raw_vs_cleaned(cases):
    """Figure comparing raw model MC output vs slice-lofted output."""
    with plt.rc_context(STYLE):
        n_cases = len(cases)
        fig = plt.figure(figsize=(16, 4.2 * n_cases), facecolor="white")
        gs = fig.add_gridspec(n_cases, 4, hspace=0.20, wspace=0.08)

        for r, case in enumerate(cases):
            phase = case["phase"]
            xyz, tissue = case["xyz"], case["tissue"]

            # Get raw meshes
            endo_raw, epi_raw = infer_raw_meshes(case)

            # Slice-lofted meshes (already stored)
            endo_clean = case["endo"]
            epi_clean = case["epi"]

            all_verts = [v for m in (endo_raw, epi_raw, endo_clean, epi_clean)
                         if m is not None and len(m.vertices) > 0
                         for v in [m.vertices]]
            all_verts.append(xyz)

            titles = [
                f"{phase} raw endo",
                f"{phase} raw epi",
                f"{phase} slice-lofted endo",
                f"{phase} slice-lofted epi",
            ]
            meshes = [endo_raw, epi_raw, endo_clean, epi_clean]
            colors = [PALETTE["endo"], PALETTE["epi"], PALETTE["endo"], PALETTE["epi"]]
            alphas = [0.90, 0.90, 0.90, 0.90]

            for c_idx in range(4):
                ax = fig.add_subplot(gs[r, c_idx], projection="3d")
                ax.set_facecolor("white")
                ax.set_axis_off()
                ax.view_init(elev=22, azim=40)

                mesh = meshes[c_idx]
                if mesh is not None and len(mesh.faces) > 0:
                    add_solid_mesh(ax, mesh, colors[c_idx], alpha=alphas[c_idx])

                # Overlay contour points
                tissue_id = 0 if c_idx % 2 == 0 else 1
                c_color = PALETTE["contour_endo"] if tissue_id == 0 else PALETTE["contour_epi"]
                pts = xyz[tissue == tissue_id]
                ax.scatter(*pts.T, c=c_color, s=1.5, alpha=0.4, depthshade=False)
                set_3d_lim(ax, all_verts)

                # Title with face count
                n_faces = len(mesh.faces) if mesh is not None else 0
                is_wt = mesh.is_watertight if (mesh is not None and len(mesh.faces) > 0) else False
                label = "raw" if c_idx < 2 else "slice-lofted"
                ax.set_title(f"{titles[c_idx]}\n{n_faces:,} faces | {'watertight' if is_wt else 'open'}",
                             fontsize=9, color="black")

        fig.suptitle(
            f"{ACDC_PATIENT_DIR.name} — raw model output vs slice-lofted reconstruction",
            fontsize=13, fontweight="bold", y=0.98
        )
        save_figure(fig, f"{ACDC_PATIENT_DIR.name}_raw_vs_cleaned")
        plt.show()


figure_raw_vs_cleaned([ed_case, es_case])

In [ ]:
def figure_raw_vs_cleaned_slices(cases, n_slices=6):
    """Slice-plane cross-sections comparing raw vs slice-lofted mesh contours.

    Shows how the raw model output has irregular boundaries at slice levels
    while the slice-lofted version produces smooth contours that pass through
    input points.
    """
    with plt.rc_context(STYLE):
        fig, axes = plt.subplots(
            len(cases), n_slices, figsize=(2.4 * n_slices, 5.2),
            facecolor="white", squeeze=False
        )

        for r, case in enumerate(cases):
            xyz, tissue = case["xyz"], case["tissue"]
            z_round = np.round(xyz[:, 2], 5)

            endo_raw, epi_raw = infer_raw_meshes(case)
            endo_clean = case["endo"]
            epi_clean = case["epi"]

            z_vals = selected_slice_z(case, n_slices)

            for c, z_val in enumerate(z_vals):
                ax = axes[r, c]
                ax.set_aspect("equal")
                ax.axis("off")

                # Raw mesh sections (dashed, thin)
                for line in mesh_section_xy(epi_raw, z_val):
                    ax.plot(line[:, 0], line[:, 1], color=PALETTE["epi"],
                            lw=0.9, ls=":", alpha=0.6, label="raw epi" if c == 0 else "")
                for line in mesh_section_xy(endo_raw, z_val):
                    ax.plot(line[:, 0], line[:, 1], color=PALETTE["endo"],
                            lw=0.9, ls=":", alpha=0.6, label="raw endo" if c == 0 else "")

                # Slice-lofted mesh sections (solid, thicker)
                for line in mesh_section_xy(epi_clean, z_val):
                    ax.plot(line[:, 0], line[:, 1], color=PALETTE["epi"],
                            lw=1.6, ls="--", label="slice-lofted epi" if c == 0 else "")
                for line in mesh_section_xy(endo_clean, z_val):
                    ax.plot(line[:, 0], line[:, 1], color=PALETTE["endo"],
                            lw=1.8, label="slice-lofted endo" if c == 0 else "")

                # Input contour points
                sm = z_round == round(float(z_val), 5)
                ax.scatter(xyz[sm & (tissue == 1), 0], xyz[sm & (tissue == 1), 1],
                           s=12, color=PALETTE["contour_epi"], alpha=0.6, zorder=5)
                ax.scatter(xyz[sm & (tissue == 0), 0], xyz[sm & (tissue == 0), 1],
                           s=12, color=PALETTE["contour_endo"], alpha=0.85, zorder=5)

                # Set view limits
                xy = xyz[sm, :2] if sm.any() else xyz[:, :2]
                ctr = xy.mean(axis=0)
                rad = max(float(np.ptp(xy, axis=0).max()) * 0.65, 1e-3)
                ax.set_xlim(ctr[0] - rad, ctr[0] + rad)
                ax.set_ylim(ctr[1] - rad, ctr[1] + rad)
                ax.set_title(f"{case['phase']} z={float(z_val):+.2f}", fontsize=8)

        # Legend
        fig.legend(handles=[
            plt.Line2D([], [], color=PALETTE["endo"], lw=0.9, ls=":", alpha=0.6, label="raw endo"),
            plt.Line2D([], [], color=PALETTE["epi"], lw=0.9, ls=":", alpha=0.6, label="raw epi"),
            plt.Line2D([], [], color=PALETTE["endo"], lw=1.8, label="slice-lofted endo"),
            plt.Line2D([], [], color=PALETTE["epi"], lw=1.6, ls="--", label="slice-lofted epi"),
            plt.Line2D([], [], marker="o", color="none", markerfacecolor=PALETTE["contour_endo"],
                       markersize=5, label="input endo"),
            plt.Line2D([], [], marker="o", color="none", markerfacecolor=PALETTE["contour_epi"],
                       markersize=5, label="input epi"),
        ], loc="lower center", ncol=6, frameon=False, fontsize=8)

        fig.suptitle(
            f"{ACDC_PATIENT_DIR.name} — slice intersections: raw (dotted) vs slice-lofted (solid)",
            fontsize=12, fontweight="bold", y=0.98
        )
        plt.tight_layout(rect=[0, 0.06, 1, 0.94])
        save_figure(fig, f"{ACDC_PATIENT_DIR.name}_raw_vs_cleaned_slices")
        plt.show()


figure_raw_vs_cleaned_slices([ed_case, es_case], n_slices=6)

## 9. Figure 2 - ACDC Input and Slice-Lofted SDF Surfaces

In [ ]:
def figure_acdc_input_and_solids(cases):
    with plt.rc_context(STYLE):
        fig = plt.figure(figsize=(16, 7.2), facecolor="white")
        gs = fig.add_gridspec(2, 4, hspace=0.25, wspace=0.15)
        all_vertices = [m.vertices for c in cases for m in (c["endo"], c["epi"]) if len(m.vertices)]
        all_vertices.append(np.zeros((1, 3), dtype=np.float32))

        for r, case in enumerate(cases):
            phase = case["phase"]
            phase_color = PALETTE["ed"] if phase == "ED" else PALETTE["es"]
            xyz, tissue = case["xyz"], case["tissue"]

            ax0 = fig.add_subplot(gs[r, 0])
            z_mid = support_mid_slice(case["data"])
            ax0.imshow(case["data"][:, :, z_mid].T, cmap=SEG_CMAP, vmin=0, vmax=3, origin="lower", interpolation="nearest")
            ax0.set_title(f"{phase} frame {case['frame']} raw ACDC slice z={z_mid}")
            ax0.axis("off")

            ax1 = fig.add_subplot(gs[r, 1])
            ax1.scatter(xyz[tissue == 0, 0], xyz[tissue == 0, 1], c=xyz[tissue == 0, 2], cmap="coolwarm", s=4, alpha=0.85, label="endo")
            ax1.scatter(xyz[tissue == 1, 0], xyz[tissue == 1, 1], c=xyz[tissue == 1, 2], cmap="coolwarm", s=4, alpha=0.45, marker="x", label="epi")
            ax1.set_aspect("equal")
            ax1.set_title(f"{phase} extracted SAX contour rings")
            ax1.set_xticks([])
            ax1.set_yticks([])
            if r == 0:
                ax1.legend(frameon=False, loc="upper right", fontsize=8)

            for c, (elev, azim, title) in enumerate([(20, 35, "solid view"), (82, 0, "top view")], start=2):
                ax = fig.add_subplot(gs[r, c], projection="3d")
                ax.set_facecolor("white")
                ax.set_axis_off()
                ax.view_init(elev=elev, azim=azim)
                add_solid_mesh(ax, case["epi"], PALETTE["epi"], alpha=0.28)
                add_solid_mesh(ax, case["endo"], PALETTE["endo"], alpha=0.92)
                ax.scatter(*xyz[tissue == 0].T, c=PALETTE["contour_endo"], s=2.0, alpha=0.55, depthshade=False)
                ax.scatter(*xyz[tissue == 1].T, c=PALETTE["contour_epi"], s=2.0, alpha=0.35, depthshade=False)
                set_3d_lim(ax, all_vertices + [xyz])
                ax.set_title(f"{phase} {title}", color=phase_color, fontsize=10)

        fig.suptitle(f"{ACDC_PATIENT_DIR.name} - final SDF INR inference on ACDC ED and ES", fontsize=13, fontweight="bold", y=0.99)
        save_figure(fig, f"{ACDC_PATIENT_DIR.name}_sdf_inference_preview")
        plt.show()

figure_acdc_input_and_solids([ed_case, es_case])


## 10. Figure 3 - Slice Plane Intersections (Slice-Lofted Only)

In [ ]:
def figure_slice_intersections(cases, n_slices=6):
    with plt.rc_context(STYLE):
        fig, axes = plt.subplots(len(cases), n_slices, figsize=(2.2 * n_slices, 4.8), facecolor="white", squeeze=False)
        for r, case in enumerate(cases):
            xyz, tissue = case["xyz"], case["tissue"]
            z_round = np.round(xyz[:, 2], 5)
            for c, z_val in enumerate(selected_slice_z(case, n_slices)):
                ax = axes[r, c]
                ax.set_aspect("equal")
                ax.axis("off")
                for line in mesh_section_xy(case["epi"], z_val):
                    ax.plot(line[:, 0], line[:, 1], color=PALETTE["epi"], lw=1.25, ls="--")
                for line in mesh_section_xy(case["endo"], z_val):
                    ax.plot(line[:, 0], line[:, 1], color=PALETTE["endo"], lw=1.5)
                sm = z_round == round(float(z_val), 5)
                ax.scatter(xyz[sm & (tissue == 1), 0], xyz[sm & (tissue == 1), 1], s=8, color=PALETTE["contour_epi"], alpha=0.55)
                ax.scatter(xyz[sm & (tissue == 0), 0], xyz[sm & (tissue == 0), 1], s=8, color=PALETTE["contour_endo"], alpha=0.85)
                xy = xyz[sm, :2] if sm.any() else xyz[:, :2]
                ctr = xy.mean(axis=0)
                rad = max(float(np.ptp(xy, axis=0).max()) * 0.65, 1e-3)
                ax.set_xlim(ctr[0] - rad, ctr[0] + rad)
                ax.set_ylim(ctr[1] - rad, ctr[1] + rad)
                ax.set_title(f"{case['phase']} z={float(z_val):+.2f}", fontsize=8)

        fig.legend(handles=[
            plt.Line2D([], [], color=PALETTE["endo"], lw=1.5, label="endo section"),
            plt.Line2D([], [], color=PALETTE["epi"], lw=1.3, ls="--", label="epi section"),
            plt.Line2D([], [], marker="o", color="none", markerfacecolor=PALETTE["contour_endo"], markersize=5, label="input endo"),
            plt.Line2D([], [], marker="o", color="none", markerfacecolor=PALETTE["contour_epi"], markersize=5, label="input epi"),
        ], loc="lower center", ncol=4, frameon=False, fontsize=9)
        fig.suptitle(f"{ACDC_PATIENT_DIR.name} - mesh section against input SAX contours", fontsize=12, fontweight="bold", y=0.98)
        plt.tight_layout(rect=[0, 0.07, 1, 0.94])
        save_figure(fig, f"{ACDC_PATIENT_DIR.name}_sdf_slice_intersections")
        plt.show()

figure_slice_intersections([ed_case, es_case], n_slices=6)


## 11. Figure 4 - Wall Thickness Solid Surface

In [ ]:
def figure_wall_thickness_surface(cases):
    all_wt = np.concatenate([c["wt_mm"][np.isfinite(c["wt_mm"])] for c in cases if len(c["wt_mm"])])
    if len(all_wt) == 0:
        print("No wall-thickness values available.")
        return
    vmin = max(0.0, float(np.percentile(all_wt, 2)))
    vmax = float(np.percentile(all_wt, 98))
    if vmax <= vmin:
        vmax = vmin + 1.0
    with plt.rc_context(STYLE):
        fig = plt.figure(figsize=(12, 5.6), facecolor="white")
        sm = None
        all_vertices = [m.vertices for c in cases for m in (c["endo"], c["epi"]) if len(m.vertices)]
        all_vertices.append(np.zeros((1, 3), dtype=np.float32))
        for i, case in enumerate(cases, start=1):
            ax = fig.add_subplot(1, len(cases), i, projection="3d")
            ax.set_facecolor("white")
            ax.set_axis_off()
            ax.view_init(elev=20, azim=35)
            add_solid_mesh(ax, case["epi"], PALETTE["epi"], alpha=0.16)
            sm = add_colored_mesh(ax, case["endo"], case["wt_mm"], cmap="viridis", vmin=vmin, vmax=vmax)
            set_3d_lim(ax, all_vertices)
            ax.set_title(f"{case['phase']} frame {case['frame']}\nmedian WT={np.nanmedian(case['wt_mm']):.1f} mm")
        if sm is not None:
            cbar = fig.colorbar(sm, ax=fig.axes, shrink=0.72, pad=0.02)
            cbar.set_label("Predicted wall thickness (mm)")
        fig.suptitle(f"{ACDC_PATIENT_DIR.name} - SDF decoder wall thickness", fontsize=12, fontweight="bold", y=0.98)
        save_figure(fig, f"{ACDC_PATIENT_DIR.name}_sdf_wall_thickness_surface")
        plt.show()

figure_wall_thickness_surface([ed_case, es_case])


## 12. Interactive Solid Model

This Plotly view is a shaded solid surface, not a wireframe. Meshes are
transformed back to world millimetres for inspection.

In [ ]:
%pip install -q plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def mesh3d_trace(mesh, color, opacity, name):
    if mesh is None or len(mesh.faces) == 0 or len(mesh.vertices) == 0:
        return None
    v = np.asarray(mesh.vertices, dtype=np.float64)
    f = np.asarray(mesh.faces, dtype=np.int64)
    return go.Mesh3d(
        x=v[:, 0], y=v[:, 1], z=v[:, 2],
        i=f[:, 0], j=f[:, 1], k=f[:, 2],
        color=color, opacity=float(opacity), name=name,
        flatshading=False,
        lighting=dict(ambient=0.45, diffuse=0.75, specular=0.35, roughness=0.35, fresnel=0.15),
        lightposition=dict(x=200, y=200, z=400),
        showscale=False, hoverinfo="name",
    )

def world_meshes(case):
    return dict(
        endo=mesh_to_world(case["endo"], case["centroid"], case["scale"], FLIP_Z),
        epi=mesh_to_world(case["epi"], case["centroid"], case["scale"], FLIP_Z),
    )

def show_solid_side_by_side(cases):
    fig = make_subplots(
        rows=1, cols=len(cases),
        specs=[[{"type": "scene"} for _ in cases]],
        subplot_titles=[f"{c['phase']} frame {c['frame']}" for c in cases],
        horizontal_spacing=0.02,
    )
    for col, case in enumerate(cases, start=1):
        wm = world_meshes(case)
        for tr in [
            mesh3d_trace(wm["epi"], PALETTE["epi"], 0.30, f"{case['phase']} epi"),
            mesh3d_trace(wm["endo"], PALETTE["endo"], 0.95, f"{case['phase']} endo"),
        ]:
            if tr is not None:
                fig.add_trace(tr, row=1, col=col)
    scene = dict(
        xaxis_title="X (mm)", yaxis_title="Y (mm)", zaxis_title="Z (mm)",
        aspectmode="data", bgcolor="white",
        xaxis=dict(backgroundcolor="white", gridcolor="#e5e5e5"),
        yaxis=dict(backgroundcolor="white", gridcolor="#e5e5e5"),
        zaxis=dict(backgroundcolor="white", gridcolor="#e5e5e5"),
        camera=dict(eye=dict(x=1.55, y=1.55, z=1.05)),
    )
    fig.update_layout(
        title=dict(text=f"{ACDC_PATIENT_DIR.name} - interactive solid SDF reconstruction", x=0.5),
        height=660, width=1200, margin=dict(l=0, r=0, t=70, b=0), showlegend=False,
    )
    for i in range(1, len(cases) + 1):
        fig.update_layout(**{f"scene{i if i > 1 else ''}": scene})
    fig.show()

show_solid_side_by_side([ed_case, es_case])


## 13. Optional Mesh Export

In [ ]:
EXPORT_PLY = False

# ── Post-processing controls ──────────────────────────────────────────────────
# Number of Taubin smoothing iterations (0 = disabled).
# Taubin smoothing alternates a forward pass (lambda_) and a backward pass
# (mu) so volume shrinkage stays negligible while high-frequency noise is
# removed.
SMOOTH_ITERATIONS  = 5       # light polish; slice anchoring is handled before MC
SMOOTH_LAMBDA      = 0.50    # forward-pass shrink factor  (0 < lambda < 1)
SMOOTH_MU          = -0.53   # backward-pass inflate factor (mu < -lambda)

# The slice-lofted SDF is already closed by construction. Leave this off by
# default; enable only for legacy/raw meshes that are genuinely open.
DO_WATERTIGHT      = False


def make_watertight(mesh: "trimesh.Trimesh") -> "trimesh.Trimesh":
    """Fill holes and repair winding only when the mesh is open.

    Steps
    -----
    1. ``fill_holes`` – stitches open boundary loops with new triangles.
    2. ``fix_normals`` – makes all face windings consistent and outward.
    3. ``remove_duplicate_faces`` / ``remove_unreferenced_vertices`` – cleans
       up any degenerate geometry introduced by the repair.

    If the mesh is already watertight the function is essentially a no-op
    (trimesh skips work it doesn't need to do).
    """
    if mesh is None or len(mesh.faces) == 0 or mesh.is_watertight:
        return mesh
    m = mesh.copy()
    try:
        trimesh.repair.fill_holes(m)          # stitch open boundary loops
    except Exception:
        pass
    try:
        trimesh.repair.fix_winding(m)         # consistent face orientation
    except Exception:
        pass
    try:
        trimesh.repair.fix_normals(m)         # outward-pointing normals
    except Exception:
        pass
    m.remove_duplicate_faces()
    m.remove_unreferenced_vertices()
    return m


def smooth_mesh(
    mesh: "trimesh.Trimesh",
    iterations: int = SMOOTH_ITERATIONS,
    lamb: float = SMOOTH_LAMBDA,
    mu: float = SMOOTH_MU,
) -> "trimesh.Trimesh":
    """Apply Taubin (lambda/mu) smoothing to reduce surface roughness.

    Taubin smoothing is preferred over plain Laplacian smoothing because
    the alternating shrink/inflate passes cancel out most of the volume
    loss, so the reconstructed shape stays faithful to the SDF prediction.

    Parameters
    ----------
    mesh       : input trimesh.Trimesh
    iterations : number of forward+backward pass pairs
    lamb       : forward-pass weight  (positive, < 1)
    mu         : backward-pass weight (negative, |mu| slightly > lamb)
    """
    if mesh is None or len(mesh.faces) == 0 or iterations <= 0:
        return mesh
    m = mesh.copy()
    try:
        trimesh.smoothing.filter_taubin(m, lamb=lamb, nu=mu, iterations=iterations)
    except Exception:
        # Fall back to Laplacian if Taubin is unavailable (older trimesh).
        try:
            trimesh.smoothing.filter_laplacian(m, iterations=iterations)
        except Exception:
            pass
    return m


def postprocess_mesh(mesh, do_watertight=DO_WATERTIGHT,
                     smooth_iter=SMOOTH_ITERATIONS):
    """Apply optional hole repair then light surface smoothing."""
    if mesh is None or len(mesh.faces) == 0:
        return mesh
    if do_watertight:
        mesh = make_watertight(mesh)
    mesh = smooth_mesh(mesh, iterations=smooth_iter)
    return mesh


def export_world_meshes(cases):
    saved = []
    for case in cases:
        wm = world_meshes(case)
        for name, mesh in wm.items():
            if len(mesh.faces) == 0:
                continue
            mesh = postprocess_mesh(mesh)
            watertight_tag = "wt" if DO_WATERTIGHT else "nowt"
            smooth_tag     = f"sm{SMOOTH_ITERATIONS}" if SMOOTH_ITERATIONS > 0 else "nosm"
            path = OUT_DIR / (
                f"{ACDC_PATIENT_DIR.name}_{case['phase']}_{name}"
                f"_sdf_final_world_mm_{watertight_tag}_{smooth_tag}.ply"
            )
            mesh.export(path)
            saved.append(path)
            wt_status = "watertight" if mesh.is_watertight else "open"
            print(f"  {name:4s} | faces={len(mesh.faces):>6,} | {wt_status}")
    return saved

if EXPORT_PLY:
    print(f"Post-processing: watertight={DO_WATERTIGHT}, smooth_iterations={SMOOTH_ITERATIONS}")
    for path in export_world_meshes([ed_case, es_case]):
        print(f"Saved {path}")
else:
    print("Set EXPORT_PLY = True to export lightly smoothed PLY files.")


## 13. AHA 17-Segment Wall Thickness Analysis

Implementation of the American Heart Association (AHA) 17-segment model for
left ventricular wall thickness quantification (Cerqueira et al., Circulation
2002). A single ED-derived anatomical reference frame is used for both ED and
ES so that segment labels remain comparable during wall-thickening analysis.
The LV endo mesh is divided along the long axis and circumferentially into 6
(basal/mid) or 4 (apical) sectors, plus the apical cap (segment 17). Wall
thickness from the SDF decoder delta field is then aggregated per segment.

In [ ]:
# --- AHA 17-segment model ------------------------------------------------------
# Reference: Cerqueira MD et al. Standardized myocardial segmentation and
# nomenclature for tomographic imaging of the heart. Circulation. 2002;105:539-542.
#
# Thesis convention used here:
#   - A single ED-derived AHA reference frame is used for both ED and ES.
#   - Long-axis coordinate t=0 is apex, t=1 is base.
#   - Circumferential angle 0 degrees is anterior; angles increase clockwise in
#     the bull's-eye plot. The septal direction is estimated from the RV label
#     when available, because the septal wall faces the RV.

AHA_SEGMENT_NAMES = {
    1: "Basal anterior", 2: "Basal anteroseptal", 3: "Basal inferoseptal",
    4: "Basal inferior", 5: "Basal inferolateral", 6: "Basal anterolateral",
    7: "Mid anterior", 8: "Mid anteroseptal", 9: "Mid inferoseptal",
    10: "Mid inferior", 11: "Mid inferolateral", 12: "Mid anterolateral",
    13: "Apical anterior", 14: "Apical septal",
    15: "Apical inferior", 16: "Apical lateral",
    17: "Apex",
}

AHA_SEGMENT_LEVEL = {
    **{s: "Basal" for s in range(1, 7)},
    **{s: "Mid" for s in range(7, 13)},
    **{s: "Apical" for s in range(13, 17)},
    17: "Apex",
}
AHA_VALID_WT_RANGE_MM = (0.1, 30.0)  # keeps numerical outliers out of segment means
AHA_LONGITUDINAL_BOUNDS = dict(apex_cap=0.05, apical=0.35, mid=0.65)


def _unit(v, fallback=None):
    v = np.asarray(v, dtype=np.float64)
    n = float(np.linalg.norm(v))
    if n > 1e-10:
        return v / n
    if fallback is None:
        fallback = np.array([1.0, 0.0, 0.0], dtype=np.float64)
    return _unit(fallback)


def _normalised_voxel_points(case, mask, max_points=12000):
    """Convert segmentation voxels to the same normalised coordinate system.

    This mirrors load_and_extract_contours(): x/y come from the NIfTI affine and
    z is slice_index * dz before centroid/scale normalisation and optional z flip.
    """
    ijk = np.argwhere(mask)
    if len(ijk) == 0:
        return np.empty((0, 3), dtype=np.float32)
    if len(ijk) > max_points:
        take = np.linspace(0, len(ijk) - 1, max_points).astype(np.int64)
        ijk = ijk[take]
    rows, cols, slc = ijk[:, 0], ijk[:, 1], ijk[:, 2]
    vox = np.column_stack([cols, rows, np.zeros(len(rows)), np.ones(len(rows))])
    world = (case["affine"] @ vox.T).T
    world[:, 2] = slc.astype(np.float64) * float(case["dz"])
    xyz = (world[:, :3] - case["centroid"]) / float(case["scale"])
    if FLIP_Z:
        xyz[:, 2] = -xyz[:, 2]
    return xyz.astype(np.float32)


def _oriented_long_axis(points, orient_points=None):
    pts = np.asarray(points, dtype=np.float64)
    pts = pts[np.all(np.isfinite(pts), axis=1)]
    if len(pts) < 3:
        return np.zeros(3, dtype=np.float64), np.array([0.0, 0.0, 1.0], dtype=np.float64)

    centroid = pts.mean(axis=0)
    _, _, vt = np.linalg.svd(pts - centroid, full_matrices=False)
    axis = _unit(vt[0], fallback=np.array([0.0, 0.0, 1.0]))

    # Orient the axis from apex to base. With FLIP_Z=True, base slices have
    # higher normalised z than apical slices, so contour z-extremes give a
    # stable orientation even if PCA flips sign.
    orient = pts if orient_points is None or len(orient_points) < 3 else np.asarray(orient_points, dtype=np.float64)
    z = orient[:, 2]
    q_lo, q_hi = np.percentile(z, [20, 80])
    apex_pts = orient[z <= q_lo]
    base_pts = orient[z >= q_hi]
    if len(apex_pts) and len(base_pts):
        apex_to_base = base_pts.mean(axis=0) - apex_pts.mean(axis=0)
        if np.dot(axis, apex_to_base) < 0:
            axis = -axis
    elif np.dot(axis, np.array([0.0, 0.0, 1.0])) < 0:
        axis = -axis
    return centroid, axis


def _rv_septal_direction(case, centroid, axis):
    data = case.get("data")
    if data is None:
        return None, "geometric fallback (no segmentation volume)"
    rv_pts = _normalised_voxel_points(case, data == LBL_RV)
    if len(rv_pts) < 10:
        return None, "geometric fallback (RV label unavailable)"

    # Use the mid-ventricular RV voxels; they are closest to the septal wall and
    # less affected by basal/apical partial coverage.
    proj = (rv_pts - centroid) @ axis
    q0, q1 = np.percentile(proj, [25, 75])
    mid_pts = rv_pts[(proj >= q0) & (proj <= q1)]
    if len(mid_pts) < 10:
        mid_pts = rv_pts

    septal = mid_pts.mean(axis=0) - centroid
    septal = septal - axis * np.dot(septal, axis)
    if np.linalg.norm(septal) < 1e-8:
        return None, "geometric fallback (RV direction degenerate)"
    return _unit(septal), "RV-centroid septal reference"


def build_aha_reference_frame(reference_case):
    """Build one anatomical frame, usually from ED, and reuse it for ED and ES."""
    endo_vertices = np.asarray(reference_case["endo"].vertices, dtype=np.float64)
    contour_pts = np.asarray(reference_case.get("xyz", endo_vertices), dtype=np.float64)
    axis_points = np.vstack([endo_vertices, contour_pts]) if len(contour_pts) else endo_vertices
    centroid, axis = _oriented_long_axis(axis_points, orient_points=contour_pts)

    septal, source = _rv_septal_direction(reference_case, centroid, axis)
    if septal is None:
        # Deterministic short-axis fallback when RV labels are missing.
        candidate = np.array([1.0, 0.0, 0.0], dtype=np.float64)
        candidate = candidate - axis * np.dot(candidate, axis)
        if np.linalg.norm(candidate) < 1e-8:
            candidate = np.array([0.0, 1.0, 0.0], dtype=np.float64)
            candidate = candidate - axis * np.dot(candidate, axis)
        septal = _unit(candidate)

    # Define x=anterior and y=septal so theta=90 degrees points toward the RV.
    e_anterior = _unit(np.cross(septal, axis))
    e_septal = _unit(np.cross(axis, e_anterior))

    proj_ref = (axis_points - centroid) @ axis
    apex_proj, base_proj = np.percentile(proj_ref, [1, 99])
    if base_proj <= apex_proj + 1e-8:
        apex_proj, base_proj = float(proj_ref.min()), float(proj_ref.max())
    if base_proj <= apex_proj + 1e-8:
        base_proj = apex_proj + 1.0

    return dict(
        centroid=centroid,
        axis=axis,
        e_anterior=e_anterior,
        e_septal=e_septal,
        apex_proj=float(apex_proj),
        base_proj=float(base_proj),
        reference_phase=reference_case["phase"],
        reference_frame=int(reference_case["frame"]),
        orientation_source=source,
    )


def project_to_aha_frame(vertices, reference):
    pts = np.asarray(vertices, dtype=np.float64)
    rel = pts - reference["centroid"]
    proj = rel @ reference["axis"]
    length = reference["base_proj"] - reference["apex_proj"]
    t = np.clip((proj - reference["apex_proj"]) / max(length, 1e-8), 0.0, 1.0)
    x = rel @ reference["e_anterior"]
    y = rel @ reference["e_septal"]
    theta = np.mod(np.arctan2(y, x), 2.0 * np.pi)
    return t, theta


def _sector_ids(theta, n_sectors):
    width = 2.0 * np.pi / int(n_sectors)
    return (np.floor(np.mod(theta + width / 2.0, 2.0 * np.pi) / width).astype(np.int32)
            % int(n_sectors))


def assign_aha_segments(vertices, wt_values, reference, valid_range=AHA_VALID_WT_RANGE_MM):
    """Assign endocardial vertices to AHA segments using a shared reference frame."""
    pts = np.asarray(vertices, dtype=np.float64)
    t, theta = project_to_aha_frame(pts, reference)
    segment_ids = np.zeros(len(pts), dtype=np.int32)

    apex_cap_t = AHA_LONGITUDINAL_BOUNDS["apex_cap"]
    apical_t = AHA_LONGITUDINAL_BOUNDS["apical"]
    mid_t = AHA_LONGITUDINAL_BOUNDS["mid"]

    apex_mask = t < apex_cap_t
    apical_mask = (t >= apex_cap_t) & (t < apical_t)
    mid_mask = (t >= apical_t) & (t < mid_t)
    basal_mask = t >= mid_t

    segment_ids[apex_mask] = 17
    segment_ids[apical_mask] = 13 + _sector_ids(theta[apical_mask], 4)
    segment_ids[mid_mask] = 7 + _sector_ids(theta[mid_mask], 6)
    segment_ids[basal_mask] = 1 + _sector_ids(theta[basal_mask], 6)

    wt = np.asarray(wt_values, dtype=np.float64)
    valid_wt = np.isfinite(wt)
    if valid_range is not None:
        lo, hi = valid_range
        valid_wt &= (wt >= lo) & (wt <= hi)

    segment_stats = {}
    for seg_id in range(1, 18):
        seg_mask = segment_ids == seg_id
        vals = wt[seg_mask & valid_wt]
        vals = vals[np.isfinite(vals)]
        n_total = int(seg_mask.sum())
        n_excluded = int(n_total - len(vals))
        if len(vals) == 0:
            segment_stats[seg_id] = dict(
                mean=np.nan, std=np.nan, median=np.nan,
                q25=np.nan, q75=np.nan, iqr=np.nan,
                min=np.nan, max=np.nan, n=0,
                n_total=n_total, n_excluded=n_excluded,
            )
        else:
            q25, q75 = np.percentile(vals, [25, 75])
            segment_stats[seg_id] = dict(
                mean=float(np.mean(vals)),
                std=float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0,
                median=float(np.median(vals)),
                q25=float(q25),
                q75=float(q75),
                iqr=float(q75 - q25),
                min=float(np.min(vals)),
                max=float(np.max(vals)),
                n=int(len(vals)),
                n_total=n_total,
                n_excluded=n_excluded,
            )
    return segment_ids, segment_stats


def aha_dataframe(segment_stats, phase):
    """Create a thesis-ready table from AHA segment statistics."""
    rows = []
    for seg_id in range(1, 18):
        s = segment_stats[seg_id]
        rows.append(dict(
            Phase=phase,
            Segment=seg_id,
            Level=AHA_SEGMENT_LEVEL[seg_id],
            Name=AHA_SEGMENT_NAMES[seg_id],
            WT_mean_mm=s["mean"],
            WT_std_mm=s["std"],
            WT_median_mm=s["median"],
            WT_Q25_mm=s["q25"],
            WT_Q75_mm=s["q75"],
            WT_IQR_mm=s["iqr"],
            WT_min_mm=s["min"],
            WT_max_mm=s["max"],
            N_valid_vertices=s["n"],
            N_total_vertices=s["n_total"],
            N_excluded_vertices=s["n_excluded"],
        ))
    return pd.DataFrame(rows)


def compute_aha_phase_comparison(ed_stats, es_stats):
    rows = []
    for seg_id in range(1, 18):
        ed = ed_stats[seg_id]["mean"]
        es = es_stats[seg_id]["mean"]
        delta = es - ed if np.isfinite(ed) and np.isfinite(es) else np.nan
        rel = delta / ed * 100.0 if np.isfinite(delta) and ed > 0.5 else np.nan
        rows.append(dict(
            Segment=seg_id,
            Level=AHA_SEGMENT_LEVEL[seg_id],
            Name=AHA_SEGMENT_NAMES[seg_id],
            ED_WT_mean_mm=ed,
            ES_WT_mean_mm=es,
            Delta_WT_mm=delta,
            Relative_thickening_pct=rel,
            ED_N_vertices=ed_stats[seg_id]["n"],
            ES_N_vertices=es_stats[seg_id]["n"],
        ))
    return pd.DataFrame(rows)


def save_table(df, name):
    path = OUT_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    print(f"Saved {path}")
    return path


# Build a single anatomical reference from ED and apply it to both phases.
aha_reference = build_aha_reference_frame(ed_case)
for case in (ed_case, es_case):
    seg_ids, seg_stats = assign_aha_segments(
        case["endo"].vertices, case["wt_mm"], aha_reference
    )
    case["aha_reference"] = aha_reference
    case["aha_segment_ids"] = seg_ids
    case["aha_segment_stats"] = seg_stats
    case["aha_df"] = aha_dataframe(seg_stats, case["phase"])

aha_combined_df = pd.concat([ed_case["aha_df"], es_case["aha_df"]], ignore_index=True)
aha_thickening_df = compute_aha_phase_comparison(
    ed_case["aha_segment_stats"], es_case["aha_segment_stats"]
)
aha_quality_df = pd.DataFrame([
    dict(
        Phase=case["phase"],
        Frame=case["frame"],
        Assigned_vertices=int((case["aha_segment_ids"] > 0).sum()),
        Empty_segments=int(sum(case["aha_segment_stats"][s]["n"] == 0 for s in range(1, 18))),
        Min_segment_vertices=int(min(case["aha_segment_stats"][s]["n"] for s in range(1, 18))),
        Excluded_WT_vertices=int(sum(case["aha_segment_stats"][s]["n_excluded"] for s in range(1, 18))),
        Reference_phase=aha_reference["reference_phase"],
        Reference_frame=aha_reference["reference_frame"],
        Orientation_source=aha_reference["orientation_source"],
    )
    for case in (ed_case, es_case)
])

patient_tag = ACDC_PATIENT_DIR.name
save_table(aha_combined_df, f"{patient_tag}_aha_wall_thickness_by_segment")
save_table(aha_thickening_df, f"{patient_tag}_aha_wall_thickening_ed_to_es")
save_table(aha_quality_df, f"{patient_tag}_aha_assignment_quality")

print("AHA 17-segment wall-thickness analysis complete.")
print(f"  Shared reference: {aha_reference['reference_phase']} frame {aha_reference['reference_frame']}")
print(f"  Orientation source: {aha_reference['orientation_source']}")
display(aha_quality_df)
display(aha_combined_df.round(2))
display(aha_thickening_df.round(2))


## 14. Figure - AHA Bull's Eye Plot (Wall Thickness)

Standard AHA 17-segment bull's eye (polar) representation of mean wall
thickness per segment. This is the canonical visualisation used in clinical
cardiac imaging literature for regional myocardial analysis.

In [ ]:
AHA_WT_CMAP = "viridis"
AHA_THICKENING_CMAP = "RdYlGn"


def bullseye_plot(segment_stats, title="", ax=None, cmap=AHA_WT_CMAP,
                  vmin=None, vmax=None, show_values=True, show_segment_ids=True,
                  value_fmt="{:.1f}", cbar_label="Wall thickness (mm)"):
    """Draw a centered AHA 17-segment bull's-eye plot.

    Segment 1 is centred at the top (anterior). Segments then proceed clockwise
    through anteroseptal, inferoseptal, inferior, inferolateral, and
    anterolateral regions, matching the assignment convention above.
    """
    values = np.array([segment_stats[i]["mean"] for i in range(1, 18)], dtype=np.float64)
    finite_vals = values[np.isfinite(values)]
    if vmin is None:
        vmin = float(np.nanmin(finite_vals)) if len(finite_vals) else 0.0
    if vmax is None:
        vmax = float(np.nanmax(finite_vals)) if len(finite_vals) else 10.0
    if vmax <= vmin:
        vmax = vmin + 1.0

    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    cmap_fn = plt.get_cmap(cmap)

    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(5.5, 5.5), subplot_kw={"projection": "polar"})
    else:
        fig = ax.figure

    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.set_yticklabels([])
    ax.set_xticklabels([])
    ax.grid(False)
    ax.spines["polar"].set_visible(False)

    ring_defs = [
        (0.66, 1.00, 6, 1),   # basal: 1-6
        (0.33, 0.66, 6, 7),   # mid: 7-12
        (0.10, 0.33, 4, 13),  # apical: 13-16
    ]

    for r_inner, r_outer, n_sectors, seg_start in ring_defs:
        d_theta = 2.0 * np.pi / n_sectors
        for k in range(n_sectors):
            seg_id = seg_start + k
            val = segment_stats[seg_id]["mean"]
            color = cmap_fn(norm(val)) if np.isfinite(val) else (0.84, 0.84, 0.84, 1.0)
            theta_start = -d_theta / 2.0 + k * d_theta
            theta_arr = np.linspace(theta_start, theta_start + d_theta, 40)
            ax.fill_between(theta_arr, r_inner, r_outer,
                            color=color, edgecolor="white", linewidth=1.1)
            if show_values:
                r_mid = (r_inner + r_outer) / 2.0
                t_mid = k * d_theta
                if np.isfinite(val):
                    label = value_fmt.format(val)
                    if show_segment_ids:
                        label = f"{seg_id}\n{label}"
                else:
                    label = f"{seg_id}\nNA" if show_segment_ids else "NA"
                ax.text(t_mid, r_mid, label, ha="center", va="center",
                        fontsize=7.5, fontweight="bold", color="black")

    val_17 = segment_stats[17]["mean"]
    color_17 = cmap_fn(norm(val_17)) if np.isfinite(val_17) else (0.84, 0.84, 0.84, 1.0)
    theta_full = np.linspace(0, 2 * np.pi, 80)
    ax.fill_between(theta_full, 0, 0.10, color=color_17, edgecolor="white", linewidth=1.1)
    if show_values:
        label = value_fmt.format(val_17) if np.isfinite(val_17) else "NA"
        if show_segment_ids:
            label = f"17\n{label}"
        ax.text(0, 0.0, label, ha="center", va="center",
                fontsize=7.5, fontweight="bold", color="black")

    # Anatomical orientation hints.
    ax.text(0, 1.08, "Anterior", ha="center", va="center", fontsize=8, color="0.35")
    ax.text(np.pi / 2, 1.08, "Septal", ha="center", va="center", fontsize=8, color="0.35")
    ax.text(np.pi, 1.08, "Inferior", ha="center", va="center", fontsize=8, color="0.35")
    ax.text(3 * np.pi / 2, 1.08, "Lateral", ha="center", va="center", fontsize=8, color="0.35")

    ax.set_ylim(0, 1.12)
    ax.set_title(title, fontsize=11, fontweight="bold", pad=18)
    return fig, ax, norm, cmap_fn


def figure_aha_bullseye(cases):
    """Side-by-side ED/ES bull's-eye plots with a shared colour scale."""
    all_means = [case["aha_segment_stats"][seg_id]["mean"]
                 for case in cases for seg_id in range(1, 18)
                 if np.isfinite(case["aha_segment_stats"][seg_id]["mean"])]
    vmin = max(0.0, float(np.percentile(all_means, 2))) if all_means else 0.0
    vmax = float(np.percentile(all_means, 98)) if all_means else 10.0

    with plt.rc_context(STYLE):
        fig, axes = plt.subplots(1, len(cases), figsize=(5.6 * len(cases), 5.9),
                                 subplot_kw={"projection": "polar"}, facecolor="white")
        if len(cases) == 1:
            axes = [axes]
        for ax, case in zip(axes, cases):
            bullseye_plot(
                case["aha_segment_stats"],
                title=f"{case['phase']} (frame {case['frame']})",
                ax=ax, cmap=AHA_WT_CMAP, vmin=vmin, vmax=vmax,
                cbar_label="Wall thickness (mm)",
            )

        sm = plt.cm.ScalarMappable(norm=mcolors.Normalize(vmin=vmin, vmax=vmax),
                                    cmap=plt.get_cmap(AHA_WT_CMAP))
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=axes, shrink=0.68, pad=0.08, aspect=30)
        cbar.set_label("Mean wall thickness (mm)", fontsize=10)

        fig.suptitle(
            f"{ACDC_PATIENT_DIR.name} - AHA 17-segment wall thickness",
            fontsize=13, fontweight="bold", y=0.98
        )
        save_figure(fig, f"{ACDC_PATIENT_DIR.name}_aha_bullseye_wt")
        plt.show()


figure_aha_bullseye([ed_case, es_case])


## 15. Figure - AHA Segment Bar Chart and Regional Comparison

Per-segment mean wall thickness with error bars (interquartile range) for ED
and ES side by side, enabling direct comparison of regional wall thickening
(systolic function). The relative wall thickening (RWT) is computed as
$(WT_{ES} - WT_{ED}) / WT_{ED}$ for each segment.

In [ ]:
def figure_aha_bar_chart(ed_stats, es_stats):
    """Thesis figure: ED/ES per-segment WT and systolic thickening."""
    seg_ids = np.arange(1, 18)
    ed_means = np.array([ed_stats[s]["mean"] for s in seg_ids], dtype=np.float64)
    es_means = np.array([es_stats[s]["mean"] for s in seg_ids], dtype=np.float64)
    ed_q25 = np.array([ed_stats[s]["q25"] for s in seg_ids], dtype=np.float64)
    ed_q75 = np.array([ed_stats[s]["q75"] for s in seg_ids], dtype=np.float64)
    es_q25 = np.array([es_stats[s]["q25"] for s in seg_ids], dtype=np.float64)
    es_q75 = np.array([es_stats[s]["q75"] for s in seg_ids], dtype=np.float64)

    ed_err = np.clip(np.array([ed_means - ed_q25, ed_q75 - ed_means]), 0, None)
    es_err = np.clip(np.array([es_means - es_q25, es_q75 - es_means]), 0, None)
    rwt = np.where((np.isfinite(ed_means)) & (np.isfinite(es_means)) & (ed_means > 0.5),
                   (es_means - ed_means) / ed_means * 100.0, np.nan)

    with plt.rc_context(STYLE):
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8.2), facecolor="white",
                                        gridspec_kw={"height_ratios": [2.1, 1], "hspace": 0.34})
        x = np.arange(17)
        w = 0.38
        ax1.bar(x - w / 2, ed_means, w, yerr=ed_err, capsize=3,
                color=PALETTE["ed"], alpha=0.88, label="Diastole (ED)", edgecolor="none",
                error_kw=dict(elinewidth=1.0, capthick=0.8))
        ax1.bar(x + w / 2, es_means, w, yerr=es_err, capsize=3,
                color=PALETTE["es"], alpha=0.88, label="Systole (ES)", edgecolor="none",
                error_kw=dict(elinewidth=1.0, capthick=0.8))

        ax1.set_xticks(x)
        ax1.set_xticklabels([str(s) for s in seg_ids], fontsize=9)
        ax1.set_ylabel("Wall thickness (mm)", fontsize=10)
        ax1.set_title(f"{ACDC_PATIENT_DIR.name} - AHA segment wall thickness (mean with IQR)",
                      fontsize=11, fontweight="bold")
        ax1.legend(frameon=False, fontsize=10, ncol=2)
        ax1.set_ylim(bottom=0)
        ax1.spines["top"].set_visible(False)
        ax1.spines["right"].set_visible(False)
        ax1.grid(axis="y", color="0.88", linewidth=0.7)

        for boundary in [5.5, 11.5, 15.5]:
            ax1.axvline(boundary, color="0.55", ls=":", lw=0.8)
            ax2.axvline(boundary, color="0.55", ls=":", lw=0.8)
        for xpos, label in [(2.5, "Basal"), (8.5, "Mid"), (13.5, "Apical"), (16.0, "Apex")]:
            ax1.text(xpos, ax1.get_ylim()[1] * 0.96, label, ha="center",
                     fontsize=8, color="0.35", style="italic")

        colors_rwt = np.where(rwt >= 0, PALETTE["es"], PALETTE["ed"])
        ax2.bar(x, rwt, 0.62, color=colors_rwt, alpha=0.85, edgecolor="none")
        ax2.axhline(0, color="black", lw=0.7)
        ax2.axhspan(30, 80, color="#91cf60", alpha=0.12, lw=0)
        ax2.set_xticks(x)
        ax2.set_xticklabels([str(s) for s in seg_ids], fontsize=9)
        ax2.set_xlabel("AHA segment", fontsize=10)
        ax2.set_ylabel("Thickening (%)", fontsize=10)
        ax2.set_title("Systolic wall thickening: (ES - ED) / ED x 100", fontsize=10)
        ax2.spines["top"].set_visible(False)
        ax2.spines["right"].set_visible(False)
        ax2.grid(axis="y", color="0.88", linewidth=0.7)

        save_figure(fig, f"{ACDC_PATIENT_DIR.name}_aha_bar_chart")
        plt.show()

    rwt_valid = rwt[np.isfinite(rwt)]
    if len(rwt_valid):
        print("\nRelative wall thickening (%) across AHA segments:")
        print(f"  Mean +/- SD: {np.mean(rwt_valid):.1f} +/- {np.std(rwt_valid, ddof=1):.1f}%")
        print(f"  Range: [{np.min(rwt_valid):.1f}, {np.max(rwt_valid):.1f}]%")


figure_aha_bar_chart(ed_case["aha_segment_stats"], es_case["aha_segment_stats"])


## 16. Final Lightweight 3D Figures - AHA and Wall Thickness

Adapted from the interactive 3D approach in `ES_dataset.ipynb`, but kept deliberately light for browser use. Only the two thesis-relevant 3D figures are generated: an AHA segment map and a wall-thickness heatmap for ED/ES final inference. Mesh faces are capped before plotting and the figures are saved as standalone HTML using CDN Plotly.


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# Keep these conservative so the browser remains responsive. Increase only for
# final high-resolution inspection on a powerful machine.
FINAL_3D_MAX_ENDO_FACES = 7000
FINAL_3D_MAX_EPI_FACES = 3500
FINAL_3D_SHOW_INLINE = False  # HTML files are saved either way; set True to show inline.

AHA_PLOTLY_COLORS = {
    1: "#e41a1c",  2: "#ff7f00",  3: "#ffd92f",  4: "#4daf4a",
    5: "#377eb8",  6: "#984ea3",  7: "#fb9a99",  8: "#fdbf6f",
    9: "#ffff99", 10: "#b2df8a", 11: "#a6cee3", 12: "#cab2d6",
    13: "#b15928", 14: "#66c2a5", 15: "#fc8d62", 16: "#8da0cb",
    17: "#808080",
}


def _aha_discrete_colorscale():
    """Discrete Plotly colorscale for integer AHA labels 1..17."""
    scale = []
    for seg_id in range(1, 18):
        lo = (seg_id - 1) / 17.0
        hi = seg_id / 17.0
        color = AHA_PLOTLY_COLORS[seg_id]
        scale.append([lo, color])
        scale.append([hi, color])
    return scale


def _light_mesh_arrays(mesh, values=None, max_faces=7000):
    """Return compact vertex/face arrays after deterministic face subsampling."""
    if mesh is None or len(mesh.faces) == 0 or len(mesh.vertices) == 0:
        empty_v = np.empty((0, 3), dtype=np.float64)
        empty_f = np.empty((0, 3), dtype=np.int64)
        return empty_v, empty_f, None, np.empty(0, dtype=np.int64)

    vertices = np.asarray(mesh.vertices, dtype=np.float64)
    faces = np.asarray(mesh.faces, dtype=np.int64)
    if len(faces) > int(max_faces):
        keep = np.linspace(0, len(faces) - 1, int(max_faces)).astype(np.int64)
        faces = faces[keep]

    used = np.unique(faces.reshape(-1))
    remap = np.full(len(vertices), -1, dtype=np.int64)
    remap[used] = np.arange(len(used), dtype=np.int64)
    faces_compact = remap[faces]
    vertices_compact = vertices[used]

    values_compact = None
    if values is not None:
        values = np.asarray(values)
        values_compact = values[used]
    return vertices_compact, faces_compact, values_compact, used


def _plotly_mesh_trace(vertices, faces, name, values=None, colorscale="Viridis",
                       cmin=None, cmax=None, color=None, opacity=1.0,
                       showscale=False, colorbar_title="", hovertext=None):
    if len(vertices) == 0 or len(faces) == 0:
        return None
    kw = dict(
        x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2],
        i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
        name=name,
        opacity=float(opacity),
        flatshading=False,
        lighting=dict(ambient=0.55, diffuse=0.75, specular=0.22, roughness=0.45, fresnel=0.12),
        lightposition=dict(x=120, y=180, z=260),
    )
    if values is not None:
        kw.update(
            intensity=np.asarray(values, dtype=np.float64),
            colorscale=colorscale,
            cmin=cmin,
            cmax=cmax,
            showscale=bool(showscale),
        )
        if showscale:
            kw["colorbar"] = dict(title=colorbar_title, thickness=14, len=0.62)
    else:
        kw.update(color=color or "#7f7f7f", showscale=False)

    if hovertext is not None:
        kw["hovertext"] = hovertext
        kw["hoverinfo"] = "text"
    else:
        kw["hoverinfo"] = "name"
    return go.Mesh3d(**kw)


def _case_world_endo_arrays(case, values=None, max_faces=FINAL_3D_MAX_ENDO_FACES):
    mesh_world = mesh_to_world(case["endo"], case["centroid"], case["scale"], FLIP_Z)
    return _light_mesh_arrays(mesh_world, values=values, max_faces=max_faces)


def _case_world_epi_arrays(case, max_faces=FINAL_3D_MAX_EPI_FACES):
    mesh_world = mesh_to_world(case["epi"], case["centroid"], case["scale"], FLIP_Z)
    return _light_mesh_arrays(mesh_world, values=None, max_faces=max_faces)


def _scene_layout():
    return dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        aspectmode="data",
        bgcolor="white",
        camera=dict(eye=dict(x=1.55, y=1.75, z=1.05)),
    )


def save_plotly_figure(fig, name, show=FINAL_3D_SHOW_INLINE):
    path = OUT_DIR / f"{name}.html"
    pio.write_html(
        fig, str(path), include_plotlyjs="cdn", full_html=True, auto_open=False,
        config={"displaylogo": False, "responsive": True},
    )
    print(f"Saved {path}")
    if show:
        fig.show(config={"displaylogo": False, "responsive": True})
    return path


def figure_final_3d_aha_segments(cases, max_faces=FINAL_3D_MAX_ENDO_FACES,
                                 show=FINAL_3D_SHOW_INLINE):
    """Interactive ED/ES endocardial AHA segment map from final inference."""
    fig = make_subplots(
        rows=1, cols=len(cases), specs=[[{"type": "scene"} for _ in cases]],
        subplot_titles=[f"{case['phase']} frame {case['frame']}" for case in cases],
        horizontal_spacing=0.02,
    )
    colorscale = _aha_discrete_colorscale()

    for col, case in enumerate(cases, start=1):
        seg_ids = np.asarray(case["aha_segment_ids"], dtype=np.int32)
        wt = np.asarray(case["wt_mm"], dtype=np.float64)
        vertices, faces, seg_plot, used = _case_world_endo_arrays(case, seg_ids, max_faces=max_faces)
        wt_plot = wt[used] if len(used) else np.empty(0)
        hover = [
            f"{case['phase']}<br>S{int(seg)}: {AHA_SEGMENT_NAMES.get(int(seg), 'Unassigned')}<br>WT={val:.2f} mm"
            for seg, val in zip(seg_plot, wt_plot)
        ]
        tr = _plotly_mesh_trace(
            vertices, faces, name=f"{case['phase']} AHA segments",
            values=seg_plot, colorscale=colorscale, cmin=0.5, cmax=17.5,
            opacity=1.0, showscale=(col == len(cases)), colorbar_title="AHA segment",
            hovertext=hover,
        )
        if tr is not None:
            fig.add_trace(tr, row=1, col=col)

    fig.update_layout(
        title=dict(text=f"{ACDC_PATIENT_DIR.name} - final inference AHA 17-segment map", x=0.5),
        height=580, width=1120, margin=dict(l=0, r=0, t=70, b=0), showlegend=False,
    )
    for i in range(1, len(cases) + 1):
        fig.update_layout(**{f"scene{i if i > 1 else ''}": _scene_layout()})
    save_plotly_figure(fig, f"{ACDC_PATIENT_DIR.name}_final_3d_aha_segments", show=show)
    return fig


def figure_final_3d_wall_thickness(cases, max_endo_faces=FINAL_3D_MAX_ENDO_FACES,
                                   max_epi_faces=FINAL_3D_MAX_EPI_FACES,
                                   show=FINAL_3D_SHOW_INLINE):
    """Interactive ED/ES wall-thickness heatmap with a lightweight epi shell."""
    all_wt = np.concatenate([
        np.asarray(case["wt_mm"], dtype=np.float64)[np.isfinite(case["wt_mm"])]
        for case in cases if len(case["wt_mm"])
    ])
    vmin = max(0.0, float(np.percentile(all_wt, 2))) if len(all_wt) else 0.0
    vmax = float(np.percentile(all_wt, 98)) if len(all_wt) else 12.0
    if vmax <= vmin:
        vmax = vmin + 1.0

    fig = make_subplots(
        rows=1, cols=len(cases), specs=[[{"type": "scene"} for _ in cases]],
        subplot_titles=[f"{case['phase']} frame {case['frame']}" for case in cases],
        horizontal_spacing=0.02,
    )

    for col, case in enumerate(cases, start=1):
        wt = np.asarray(case["wt_mm"], dtype=np.float64)
        seg_ids = np.asarray(case["aha_segment_ids"], dtype=np.int32)

        epi_v, epi_f, _, _ = _case_world_epi_arrays(case, max_faces=max_epi_faces)
        epi_tr = _plotly_mesh_trace(
            epi_v, epi_f, name=f"{case['phase']} epi reference",
            color="#d0d0d0", opacity=0.18,
        )
        if epi_tr is not None:
            fig.add_trace(epi_tr, row=1, col=col)

        endo_v, endo_f, wt_plot, used = _case_world_endo_arrays(case, wt, max_faces=max_endo_faces)
        seg_plot = seg_ids[used] if len(used) else np.empty(0, dtype=np.int32)
        hover = [
            f"{case['phase']}<br>WT={val:.2f} mm<br>S{int(seg)}: {AHA_SEGMENT_NAMES.get(int(seg), 'Unassigned')}"
            for val, seg in zip(wt_plot, seg_plot)
        ]
        endo_tr = _plotly_mesh_trace(
            endo_v, endo_f, name=f"{case['phase']} wall thickness",
            values=wt_plot, colorscale="Viridis", cmin=vmin, cmax=vmax,
            opacity=1.0, showscale=(col == len(cases)), colorbar_title="WT (mm)",
            hovertext=hover,
        )
        if endo_tr is not None:
            fig.add_trace(endo_tr, row=1, col=col)

    fig.update_layout(
        title=dict(text=f"{ACDC_PATIENT_DIR.name} - final inference wall-thickness heatmap", x=0.5),
        height=580, width=1120, margin=dict(l=0, r=0, t=70, b=0), showlegend=False,
    )
    for i in range(1, len(cases) + 1):
        fig.update_layout(**{f"scene{i if i > 1 else ''}": _scene_layout()})
    save_plotly_figure(fig, f"{ACDC_PATIENT_DIR.name}_final_3d_wall_thickness", show=show)
    return fig


fig_final_3d_aha = figure_final_3d_aha_segments([ed_case, es_case])
fig_final_3d_wt = figure_final_3d_wall_thickness([ed_case, es_case])


## 17. AHA Summary Metrics and Statistical Report

Quantitative metrics per AHA level (basal, mid, apical, apex) and global
statistics including:
- Mean ± SD wall thickness per level and phase
- Coefficient of variation as a measure of regional heterogeneity  
- Global relative wall thickening
- Myocardial mass estimate from the SDF volumes

In [ ]:
def aha_level_stats(segment_stats):
    """Aggregate WT statistics by AHA level using segment means."""
    levels = {
        "Basal (1-6)": range(1, 7),
        "Mid (7-12)": range(7, 13),
        "Apical (13-16)": range(13, 17),
        "Apex (17)": [17],
    }
    rows = []
    for level_name, seg_range in levels.items():
        means = np.array([segment_stats[s]["mean"] for s in seg_range], dtype=np.float64)
        means = means[np.isfinite(means)]
        all_n = sum(segment_stats[s]["n"] for s in seg_range)
        rows.append(dict(
            Level=level_name,
            Mean_WT_mm=float(np.mean(means)) if len(means) else np.nan,
            SD_WT_mm=float(np.std(means, ddof=1)) if len(means) > 1 else 0.0 if len(means) == 1 else np.nan,
            Min_WT_mm=float(np.min(means)) if len(means) else np.nan,
            Max_WT_mm=float(np.max(means)) if len(means) else np.nan,
            CoV_pct=float(np.std(means, ddof=1) / np.mean(means) * 100) if len(means) > 1 and np.mean(means) > 0 else np.nan,
            N_valid_vertices=int(all_n),
        ))
    return pd.DataFrame(rows)


def myocardial_mass_g(epi_vol_ml, endo_vol_ml, density_g_per_ml=1.05):
    """Estimate LV myocardial mass from myocardial volume and density."""
    if np.isnan(epi_vol_ml) or np.isnan(endo_vol_ml) or epi_vol_ml < endo_vol_ml:
        return np.nan
    return (epi_vol_ml - endo_vol_ml) * density_g_per_ml


print("=" * 72)
print(f"  AHA 17-Segment Wall Thickness Report - {ACDC_PATIENT_DIR.name}")
print("=" * 72)

level_tables = []
for case in (ed_case, es_case):
    print(f"\n{'-' * 30} {case['phase']} (frame {case['frame']}) {'-' * 30}")
    level_df = aha_level_stats(case["aha_segment_stats"])
    level_df.insert(0, "Phase", case["phase"])
    level_tables.append(level_df)
    display(level_df.round(2))

aha_level_df = pd.concat(level_tables, ignore_index=True)
save_table(aha_level_df, f"{ACDC_PATIENT_DIR.name}_aha_wall_thickness_by_level")

print(f"\n{'=' * 72}")
print("  Global cardiac metrics")
print(f"{'=' * 72}")

global_rows = []
for case in (ed_case, es_case):
    wt = np.asarray(case["wt_mm"], dtype=np.float64)
    wt_valid = wt[np.isfinite(wt)]
    wt_valid = wt_valid[(wt_valid >= AHA_VALID_WT_RANGE_MM[0]) & (wt_valid <= AHA_VALID_WT_RANGE_MM[1])]
    endo_vol = mesh_volume_ml(case["endo"], case["scale"])
    epi_vol = mesh_volume_ml(case["epi"], case["scale"])
    global_rows.append(dict(
        Phase=case["phase"],
        Frame=case["frame"],
        Global_mean_WT_mm=float(np.mean(wt_valid)) if len(wt_valid) else np.nan,
        Global_SD_WT_mm=float(np.std(wt_valid, ddof=1)) if len(wt_valid) > 1 else np.nan,
        Global_median_WT_mm=float(np.median(wt_valid)) if len(wt_valid) else np.nan,
        Endo_vol_mL=endo_vol,
        Epi_vol_mL=epi_vol,
        Myo_vol_mL=epi_vol - endo_vol if not (np.isnan(epi_vol) or np.isnan(endo_vol)) else np.nan,
        Myo_mass_g=myocardial_mass_g(epi_vol, endo_vol),
        Mesh_endo_watertight=bool(case["endo"].is_watertight) if len(case["endo"].faces) else False,
        Mesh_epi_watertight=bool(case["epi"].is_watertight) if len(case["epi"].faces) else False,
    ))

global_df = pd.DataFrame(global_rows)
save_table(global_df, f"{ACDC_PATIENT_DIR.name}_global_metrics")
display(global_df.round(2))

ed_endo_vol = global_df.loc[global_df["Phase"] == "ED", "Endo_vol_mL"].values[0]
es_endo_vol = global_df.loc[global_df["Phase"] == "ES", "Endo_vol_mL"].values[0]
if not (np.isnan(ed_endo_vol) or np.isnan(es_endo_vol)) and ed_endo_vol > 0:
    ef = (ed_endo_vol - es_endo_vol) / ed_endo_vol * 100.0
    print(f"\n  Ejection fraction (EF) = {ef:.1f}%")
    print(f"    EDV = {ed_endo_vol:.1f} mL | ESV = {es_endo_vol:.1f} mL | SV = {ed_endo_vol - es_endo_vol:.1f} mL")
else:
    print("\n  Ejection fraction: could not compute from degenerate mesh volumes")

ed_global_wt = global_df.loc[global_df["Phase"] == "ED", "Global_mean_WT_mm"].values[0]
es_global_wt = global_df.loc[global_df["Phase"] == "ES", "Global_mean_WT_mm"].values[0]
if np.isfinite(ed_global_wt) and np.isfinite(es_global_wt) and ed_global_wt > 0:
    gwt = (es_global_wt - ed_global_wt) / ed_global_wt * 100.0
    print(f"  Global wall thickening = {gwt:.1f}%")
    print(f"    ED mean WT = {ed_global_wt:.2f} mm | ES mean WT = {es_global_wt:.2f} mm")

print("\n  Regional heterogeneity (CoV of AHA segment means):")
for case in (ed_case, es_case):
    means = np.array([case["aha_segment_stats"][s]["mean"] for s in range(1, 18)], dtype=np.float64)
    means = means[np.isfinite(means)]
    if len(means) > 1 and np.mean(means) > 0:
        cov = np.std(means, ddof=1) / np.mean(means) * 100.0
        print(f"    {case['phase']}: CoV = {cov:.1f}% (N={len(means)} segments)")


## 18. Figure - Wall Thickening Bull's Eye (ED → ES)

Bull's eye plot of the relative wall thickening per AHA segment ($\Delta WT\%$).
The colour bands are descriptive reference ranges for visual screening only;
they are not used as diagnostic labels.

In [ ]:
def compute_thickening_stats(ed_stats, es_stats):
    """Return bull's-eye-compatible stats for relative ED-to-ES thickening."""
    thickening_stats = {}
    for seg_id in range(1, 18):
        ed_mean = ed_stats[seg_id]["mean"]
        es_mean = es_stats[seg_id]["mean"]
        if np.isfinite(ed_mean) and np.isfinite(es_mean) and ed_mean > 0.5:
            rwt = (es_mean - ed_mean) / ed_mean * 100.0
        else:
            rwt = np.nan
        thickening_stats[seg_id] = dict(
            mean=rwt, std=0.0, median=rwt,
            q25=rwt, q75=rwt, iqr=0.0, min=rwt, max=rwt, n=1,
            n_total=1, n_excluded=0,
        )
    return thickening_stats


def classify_thickening(value):
    if not np.isfinite(value):
        return "Not available"
    if value < 10:
        return "Akinetic/dyskinetic"
    if value < 30:
        return "Hypokinetic"
    if value <= 80:
        return "Reference range"
    return "Hyperkinetic"


def figure_thickening_bullseye(ed_case, es_case):
    """Bull's-eye of relative wall thickening from diastole to systole."""
    thickening = compute_thickening_stats(
        ed_case["aha_segment_stats"], es_case["aha_segment_stats"]
    )
    values = np.array([thickening[s]["mean"] for s in range(1, 18)], dtype=np.float64)
    class_df = pd.DataFrame([
        dict(Segment=s, Name=AHA_SEGMENT_NAMES[s], Relative_thickening_pct=thickening[s]["mean"],
             Category=classify_thickening(thickening[s]["mean"]))
        for s in range(1, 18)
    ])
    save_table(class_df, f"{ACDC_PATIENT_DIR.name}_aha_thickening_categories")

    with plt.rc_context(STYLE):
        fig = plt.figure(figsize=(9, 5.6), facecolor="white")
        ax_be = fig.add_subplot(1, 2, 1, projection="polar")
        ax_legend = fig.add_subplot(1, 2, 2)

        bullseye_plot(
            thickening,
            title="Relative wall thickening (%)\n(ES - ED) / ED x 100",
            ax=ax_be, cmap=AHA_THICKENING_CMAP, vmin=-20, vmax=120,
            value_fmt="{:.0f}", cbar_label="Thickening (%)",
        )

        sm = plt.cm.ScalarMappable(norm=mcolors.Normalize(vmin=-20, vmax=120),
                                    cmap=plt.get_cmap(AHA_THICKENING_CMAP))
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax_be, shrink=0.72, pad=0.10)
        cbar.set_label("Thickening (%)", fontsize=9)

        ax_legend.axis("off")
        ax_legend.set_xlim(0, 1)
        ax_legend.set_ylim(0, 1)
        interpretation = [
            ("Akinetic / dyskinetic", "< 10%", "#d73027"),
            ("Hypokinetic", "10-30%", "#fc8d59"),
            ("Reference range", "30-80%", "#91cf60"),
            ("Hyperkinetic", "> 80%", "#1a9850"),
        ]
        for idx, (label, range_str, color) in enumerate(interpretation):
            y = 0.82 - idx * 0.17
            ax_legend.add_patch(plt.Rectangle((0.05, y - 0.035), 0.12, 0.07,
                                              facecolor=color, edgecolor="none", alpha=0.85))
            ax_legend.text(0.22, y, f"{label}\n{range_str}", fontsize=9, va="center")
        ax_legend.set_title("Descriptive bands", fontsize=9, fontweight="bold")

        vals = values[np.isfinite(values)]
        if len(vals):
            counts = class_df["Category"].value_counts().to_dict()
            summary = f"N={len(vals)} segments\n"
            summary += f"Mean={np.mean(vals):.1f}%\n"
            summary += f"Range={np.min(vals):.1f} to {np.max(vals):.1f}%\n"
            summary += "\n".join(f"{k}: {v}" for k, v in counts.items())
            ax_legend.text(0.05, 0.05, summary, fontsize=8, va="bottom", color="0.35")

        fig.suptitle(f"{ACDC_PATIENT_DIR.name} - systolic wall thickening (AHA 17 segments)",
                     fontsize=12, fontweight="bold", y=0.98)
        plt.tight_layout(rect=[0, 0, 1, 0.94])
        save_figure(fig, f"{ACDC_PATIENT_DIR.name}_aha_thickening_bullseye")
        plt.show()


figure_thickening_bullseye(ed_case, es_case)
